# Ration Early or Run Dry?

## A reservoir trade-off in four TaqSim blocks

Dr. Tobias Siegfried, hydrosolutions GmbH · CC BY 4.0

<p><img src="https://hydrosolutions.github.io/zarafshan-taqsim-course/images/hydrosolutions_logo.png" alt="hydrosolutions" width="220"></p>
<p><b>hydrosolutions GmbH</b> · Venusstrasse 29, 8050 Zürich, Switzerland · <a href="https://www.hydrosolutions.ch">www.hydrosolutions.ch</a> · hs@hydrosolutions.ch · +41 43 535 05 80</p>

# The question

A reservoir fills in winter. A farm takes water from it in summer. In most years there is enough. In a dry year
there is not, and the operator has two ways to run the reservoir:

- **Deliver in full until the reservoir is empty.** The farm gets everything it asks for, for as long as possible.
  If the water runs out, the last weeks of summer get almost nothing.
- **Ration early.** When the reservoir is getting low, deliver only part of what the farm asks for. The water lasts
  longer, but the farm is short more often, including in years when the water would have lasted anyway.

Which is better? It depends on what you count. This notebook builds the smallest model that shows it: four TaqSim
blocks, two knobs, two goals. Then it lets the optimizer search.

**What you will take away**

1. How to build and run a small water system in TaqSim, block by block.
2. Why two reasonable goals can pull in opposite directions, and how to see every sensible compromise at once.
3. How to hand the search to the optimizer, and how to read what it returns.

> **The technical words.** Engineers call "deliver in full until empty" the *standard operating policy* and
> "ration early" *hedging*. A plan that no other plan beats on every goal is called *Pareto-optimal*, and the set
> of such plans is the *Pareto front*. This notebook says "unbeaten plans".

# The system in four blocks

TaqSim describes a water system as a **network**: nodes that hold, use or pass on water, joined by edges along
which water flows. Ours will have four nodes and three edges. We start with three, the river, the farm and the
outlet, and add the reservoir once we have seen what goes wrong without it.

In [ ]:
# @title Setup: run this cell first
# ── Setup: run this cell first. In Colab it takes about a minute. ──────────────────────────────────
# It fetches the course folder, installs TaqSim, and tells Python where the course code lives.
import os
import sys
import subprocess

if not os.path.exists("src/zarafshan_taqsim"):  # not inside the course folder yet (the normal case in Colab)
    if not os.path.exists("zarafshan-taqsim-course"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/hydrosolutions/zarafshan-taqsim-course.git"], check=True)
    os.chdir("zarafshan-taqsim-course")
# A runtime that ran this cell earlier may hold an older copy of the course: bring the folder up to date and
# forget any course module Python has already loaded, so that the current code is what gets imported.
if os.path.exists(".git"):
    subprocess.run(["git", "pull", "-q", "--ff-only"], check=False)
for name in [m for m in sys.modules if m == "zarafshan_taqsim" or m.startswith("zarafshan_taqsim.")]:
    del sys.modules[name]
try:
    import taqsim  # noqa: F401  (already installed when you work on your own computer)
except ImportError:
    %pip install -q deap ctrl-freak pymoo
    %pip install -q --no-deps "git+https://github.com/hydrosolutions/taqsim.git@v0.1.4"
sys.path[:0] = [os.path.abspath("src")]
print("Ready. Working folder:", os.getcwd())


In [ ]:
# @title Code: setup
#| label: setup
# Standard tools
import inspect  # to show the source code of a function or class inside this notebook
import logging

import pandas as pd  # tables
from IPython.display import HTML, Markdown  # to show formatted text and HTML as cell output

# TaqSim itself: the four kinds of node we use, the edge that joins them, and the system that holds them
from taqsim import Demand, Edge, Sink, Source, Storage, TimeSeries, WaterSystem
from taqsim.node import NoLoss  # a loss rule that loses nothing (reservoirs need one)
from taqsim.node.events import DeficitRecorded  # the event a Demand node records when it is short
from taqsim.time import Frequency  # the length of one simulation step (monthly here)

# The helpers written for this coursebook (src/zarafshan_taqsim/rationing_coursebook.py)
from zarafshan_taqsim.rationing_coursebook import (
    CAPACITY_MM3,
    EVAPORATION_SHARE_EXAMPLE,
    MONTHLY_NEED_MM3,
    NO_RATIONING,
    PAPER_REGIME,
    START_STORAGE_MM3,
    SUMMER_NEED_MM3,
    WINTER_NEED_MM3,
    YEARS,
    Plan,
    RationingRelease,
    SUMMER_PLAN_FOR_WINTER,
    WINTER_FIELDS_TOTAL_MM3,
    WINTER_IN_FULL,
    SummerEvaporation,
    WinterPlan,
    WinterRelease,
    drought_window_start,
    explorer_html,
    explorer_payload,
    front_of,
    objectives,
    optimise_plans,
    plan_grid,
    forecast_value,
    least_winter_shortage,
    plot_forecast_skill,
    plot_forecast_value,
    plot_front_shift,
    plot_inflows,
    plot_network,
    plot_regime_comparison,
    plot_runs,
    plot_score_maps,
    plot_shortage_years,
    plot_system_sketch,
    plot_tradeoff,
    plot_winter_decisions,
    plot_winter_fronts,
    score_by_skill,
    score_plans,
    score_winter_plans,
    season_statistics,
    season_totals,
    seasonal_forecast,
    simulate_plan,
    simulate_winter,
    simulate_without_reservoir,
    synthetic_inflows,
    winter_grid,
    winter_plan_within,
)

logging.basicConfig(level=logging.WARNING)  # only show warnings and errors, not progress messages
pd.set_option("display.precision", 2)  # two decimals in tables

LANG = "en"  # the web page shows all three languages and lets you switch; a notebook is built for one language


def say(en: str, uz: str, ru: str) -> Markdown:
    """Text in three languages: all three on the web page (the switcher picks one), one in a notebook."""
    if LANG:
        return Markdown({"en": en, "uz": uz, "ru": ru}[LANG])
    blocks = (("en", en), ("uz", uz), ("ru", ru))
    return Markdown("\n\n".join(f"::: {{.lang-{code}}}\n{text}\n:::" for code, text in blocks))


def show_source(thing) -> Markdown:
    """Show the source code of a function or class as a code block."""
    return Markdown(f"```python\n{inspect.getsource(thing)}```")

In [ ]:
# @title Code: fig-network
#| label: fig-network
#| fig-cap: "The model as a network. Each node is one TaqSim block; each edge is named after the nodes it joins."
#| echo: false
_ = plot_network()

- **river** is a `Source`: it puts water into the network each month, from a list of numbers.
- **reservoir** is a `Storage`: it keeps water up to its capacity, loses some (if we tell it to), and releases
  some according to a rule. This is where the decisions are made.
- **farm** is a `Demand`: it asks for a given amount each month and records how much it did not get.
- **downstream** is a `Sink`: whatever the farm does not use leaves the network here.

In [ ]:
# @title Code: fig-system
#| label: fig-system
#| fig-cap: "The same four blocks with their numbers. Water moves from left to right."
#| echo: false
_ = plot_system_sketch()

The numbers come from a classic paper on reservoir performance (Hashimoto, Stedinger and Loucks, 1982, p. 17 and
Table 1), with one change. The paper's river peaks in winter. Rivers in Central Asia are fed by snow and glacier
melt: they peak in summer and are lowest in winter. So we swap the paper's two seasons and keep everything else.
[The comparison section](#sec-paper-regime) runs the paper's own river alongside ours. Volumes are in million
cubic metres (Mm³).

| Quantity | Value |
|---|---|
| Reservoir capacity | 40 Mm³ |
| What the farm needs in summer | 45 Mm³ (7.5 Mm³ a month for six months) |
| What the farm needs in winter | 5 Mm³ (0.83 Mm³ a month for six months): a small off-season demand, taken from the paper |
| River inflow in winter | 25 Mm³ on average, standard deviation 10 Mm³ (the paper: 40 and 15) |
| River inflow in summer | 40 Mm³ on average, standard deviation 15 Mm³ (the paper: 25 and 10) |

The summer need (45) is larger than the average summer inflow (40), so the farm depends on water stored in winter.
The reservoir (40) is smaller than the need, and it can only be filled from the low winter flow, so there is
little room for error.

A model year is six winter months followed by six summer months; no calendar dates are attached. Read it as the
hydrological year used in the Zarafshan basin: winter is October to March, summer (the irrigation season) April
to September. Within a season nothing changes from month to month: each season's inflow is spread evenly over
its six months, and so is the farm's need. The only seasonality is winter against summer, as in the paper. The real
monthly shape of the Zarafshan, with its July peak, appears in the sixteen-blocks game and in the simulation
coursebook.

## The river

We do not have a hundred years of measurements, so we make a river: one hundred years of winter and summer inflows
with the averages and spread from the table above. A dry season tends to be followed by another dry season, as in
real rivers.

In [ ]:
# @title Code: river
#| label: river
inflows = synthetic_inflows()  # 100 years x 12 months = 1 200 numbers, each a month's inflow in Mm³

# Check the river against the numbers we asked for: averages, spread and the link from one season to the next
season_statistics(inflows).style.hide(axis="index").format(precision=2)

`synthetic_inflows()` always gives the same river (it uses a fixed random seed), so every result in this notebook
can be reproduced. The table compares the river's statistics with the ones we asked for; they agree to within a
few percent.

In [ ]:
# @title Code: fig-river
#| label: fig-river
#| fig-cap: "One hundred synthetic years. In many years the summer inflow is far below what the farm needs."
#| echo: false
_ = plot_inflows(inflows)

> **Where this differs from the paper.** (1) The seasons are swapped, as explained above. (2) Each season's
> inflow is spread evenly over six monthly steps. (3) The paper derives its operating rules mathematically; we use
> a simple rule with two knobs. (4) The link between seasons is applied to the logarithms of the flows, which is
> an assumption. The numbers in this notebook are therefore our own, not a reproduction of the paper's.

# Before the reservoir: a river and a farm

Start without the reservoir. The farm sits on the river and takes what the river brings; what it does not use
flows on. Three blocks, two edges, and nothing to decide: there is no rule, because there is nothing to store.

In [ ]:
# @title Code: fig-river-farm
#| label: fig-river-farm
#| fig-cap: "The system without a reservoir: three blocks, two edges."
#| echo: false
_ = plot_network(reservoir=False)

Building a TaqSim model always has the same four steps: create an empty system, add the nodes, add the edges,
check and run. Here are all four for the three blocks.

In [ ]:
# @title Code: river-farm
#| label: river-farm
# 1. An empty system. One simulation step is one month.
river_farm = WaterSystem(frequency=Frequency.MONTHLY)

# 2. Three nodes: the river gives water, the farm asks for it, the outlet takes what is left.
river_farm.add_node(Source(id="river", inflow=TimeSeries(values=list(inflows))))
river_farm.add_node(Demand(id="farm", requirement=TimeSeries(values=list(MONTHLY_NEED_MM3) * YEARS)))
river_farm.add_node(Sink(id="downstream"))

# 3. Two edges, each named after the nodes it joins.
for source, target in [("river", "farm"), ("farm", "downstream")]:
    river_farm.add_edge(Edge(id=f"{source}_to_{target}", source=source, target=target))

# 4. Check the network, then run all 1 200 months.
river_farm.validate()
river_farm.simulate(len(inflows))

# The farm records a DeficitRecorded event in every month it gets less than it needs; add up the missing water
not_delivered = sum(event.deficit for event in river_farm.nodes["farm"].events_of_type(DeficitRecorded))
print(f"Water not delivered: {not_delivered / YEARS:.1f} Mm³ per year, on average")

In [ ]:
# @title Code: fig-why-a-reservoir
#| label: fig-why-a-reservoir
#| fig-cap: "The worst month of each year without a reservoir (top) and with a reservoir of 40 Mm³ that delivers in full until it is empty (bottom)."
#| echo: false
without_reservoir = simulate_without_reservoir(inflows)  # the three blocks above, run by a helper
with_reservoir = simulate_plan(inflows, NO_RATIONING)  # the four blocks built in the next section
_ = plot_shortage_years(
    {
        "River and farm, no reservoir": without_reservoir,
        "With a reservoir of 40 Mm³, delivered in full until empty": with_reservoir,
    }
)

In [ ]:
# @title Code: why-a-reservoir
#| label: why-a-reservoir
#| echo: false
a, b = without_reservoir.scores, with_reservoir.scores
say(
    f"That is a lot: the farm asks for {SUMMER_NEED_MM3 + WINTER_NEED_MM3:g} Mm³ a year and misses "
    f"**{a.mean_shortage:.1f}** of them, and it is short in **{a.years_short} of {YEARS} years**. The reason is not "
    f"a lack of water but its timing: the farm needs {SUMMER_NEED_MM3:g} Mm³ in summer, the river brings 40 on "
    f"average, and the 25 Mm³ that arrive in winter flow past the farm unused. The lower panel shows what a "
    f"reservoir does. A reservoir of {CAPACITY_MM3:g} Mm³ carries winter water into summer: the short years fall "
    f"from {a.years_short} to **{b.years_short}**, the water missed from {a.mean_shortage:.1f} to "
    f"**{b.mean_shortage:.1f} Mm³ a year**. This is what reservoirs are for.\n\n"
    f"Two things it does not do. It does not change the single worst month ({a.worst_month:.0f} % short in both "
    f"panels): in the driest summer the reservoir is already empty, because it holds only {CAPACITY_MM3:g} Mm³ and "
    f"can only fill from the low winter flow. And it brings a decision with it, how to release the water it holds, "
    f"which the river alone never asked of us. The rest of this coursebook is about that decision.",
    f"Bu koʻp: xoʻjalik yiliga {SUMMER_NEED_MM3 + WINTER_NEED_MM3:g} mln m³ soʻraydi va shundan "
    f"**{a.mean_shortage:.1f}** ini olmaydi, {YEARS} yildan **{a.years_short} tasida** taqchillik koʻradi. Sabab suv "
    f"yetishmasligida emas, uning vaqtida: xoʻjalikka yozda {SUMMER_NEED_MM3:g} mln m³ kerak, daryo oʻrtacha 40 "
    f"keltiradi, qishda kelgan 25 mln m³ esa xoʻjalik yonidan foydalanilmay oqib oʻtadi. Pastki panel suv ombori "
    f"nima qilishini koʻrsatadi. {CAPACITY_MM3:g} mln m³ li suv ombori qishki suvni yozga olib oʻtadi: taqchil yillar "
    f"{a.years_short} tadan **{b.years_short} taga**, olinmagan suv yiliga {a.mean_shortage:.1f} dan "
    f"**{b.mean_shortage:.1f} mln m³ ga** tushadi. Suv omborlari ana shu uchun quriladi.\n\n"
    f"U qilmaydigan ikki narsa bor. U eng yomon oyni oʻzgartirmaydi (ikkala panelda ham {a.worst_month:.0f} % "
    f"taqchillik): eng qurgʻoqchil yozda ombor allaqachon boʻsh, chunki u bor-yoʻgʻi {CAPACITY_MM3:g} mln m³ sigʻdiradi "
    f"va faqat kam suvli qishki oqimdan toʻlishi mumkin. Va u oʻzi bilan bir qarorni olib keladi: saqlagan suvini "
    f"qanday chiqarish kerak, buni daryoning oʻzi hech qachon bizdan soʻramagan edi. Bu darslikning qolgan qismi ana "
    f"shu qaror haqida.",
    f"Это много: хозяйство просит {SUMMER_NEED_MM3 + WINTER_NEED_MM3:g} млн м³ в год и недополучает "
    f"**{a.mean_shortage:.1f}** из них, а дефицит испытывает в **{a.years_short} из {YEARS} лет**. Причина не в "
    f"нехватке воды, а в её сроках: хозяйству нужно {SUMMER_NEED_MM3:g} млн м³ летом, река приносит в среднем 40, а "
    f"25 млн м³, приходящие зимой, протекают мимо хозяйства без пользы. Нижняя панель показывает, что делает "
    f"водохранилище. Водохранилище на {CAPACITY_MM3:g} млн м³ переносит зимнюю воду в лето: число дефицитных лет "
    f"падает с {a.years_short} до **{b.years_short}**, недополученная вода с {a.mean_shortage:.1f} до "
    f"**{b.mean_shortage:.1f} млн м³ в год**. Для этого водохранилища и строят.\n\n"
    f"Двух вещей оно не делает. Оно не меняет самый худший месяц ({a.worst_month:.0f} % дефицита на обеих панелях): в "
    f"самое сухое лето водохранилище уже пусто, потому что вмещает лишь {CAPACITY_MM3:g} млн м³ и наполняется только "
    f"из скудного зимнего стока. И оно приносит с собой решение, как выпускать накопленную воду, которого река сама "
    f"по себе от нас никогда не требовала. Остальная часть этого учебника об этом решении.",
)

# Build it in TaqSim

## The rule with two knobs

**What a class is, in plain words.** A program like TaqSim is built from *objects*: a reservoir, a farm, a rule.
A **class** describes one kind of object: what it remembers and what it can do. Think of a printed form with a few
fields to fill in and a short instruction on the back. The class is the blank form; every object made from it is one
filled-in copy. Our rule class remembers two numbers, the trigger and the ration (its *fields*), and knows how to do
one thing, `release(...)`, which TaqSim asks it to do every month (its *method*). Writing
`RationingRelease(trigger=16, ration=0.8)` fills in one copy of the form. `Plan`, `Storage` and `Demand` below are
other forms of the same kind. That is all "object-oriented" means here: the model is a handful of such forms that
pass water to each other. You need nothing more to read or change this notebook.

A TaqSim reservoir needs a *release rule*: a small class that says how much water to let out each month. Ours has
two knobs.

- **Trigger** (Mm³): rationing starts when the reservoir holds less than this.
- **Ration** (0 to 1): the share of the summer need that is then delivered.

In [ ]:
# @title Code: rule-source
#| label: rule-source
#| echo: false
show_source(RationingRelease)

Reading the class from the top:

- `@dataclass(frozen=True)` makes it a small, unchangeable record: it holds its two knobs and nothing else.
- `__params__` names the knobs the optimizer may turn, and `__bounds__` says between which values.
- `trigger` and `ration` are the knobs, with their defaults (0 and 1: no rationing).
- `release(...)` is called by TaqSim once per month. It gets the reservoir (`node`, from which it reads the
  current storage), this month's inflow and the time step `t`, and returns the volume to release in Mm³.

A trigger of 0 (or a ration of 1) means no rationing at all.

## Wire the four blocks

The same four steps as for the three blocks: create an empty system, add the nodes, add the edges, check and run.
The reservoir goes between the river and the farm, and it needs the rule from the section above.

In [ ]:
# @title Code: wire
#| label: wire
# 1. An empty system. One simulation step is one month.
system = WaterSystem(frequency=Frequency.MONTHLY)

# 2. The four nodes. Each has an id (its name in the network) and the settings its kind needs.
system.add_node(
    Source(
        id="river",
        inflow=TimeSeries(values=list(inflows)),  # one value per month, Mm³
    )
)
system.add_node(
    Storage(
        id="reservoir",
        capacity=CAPACITY_MM3,  # 40 Mm³; more than this spills
        initial_storage=START_STORAGE_MM3,  # 20 Mm³ in store when the simulation starts
        release_policy=RationingRelease(trigger=0.0, ration=1.0),  # our rule; these knobs mean no rationing
        loss_rule=NoLoss(),  # no evaporation for now
    )
)
system.add_node(
    Demand(
        id="farm",
        requirement=TimeSeries(values=list(MONTHLY_NEED_MM3) * YEARS),  # the 12 monthly needs, repeated 100 times
    )
)
system.add_node(Sink(id="downstream"))

# 3. The three edges. An edge is named after the two nodes it joins and carries whatever its source lets out.
for source, target in [("river", "reservoir"), ("reservoir", "farm"), ("farm", "downstream")]:
    system.add_edge(Edge(id=f"{source}_to_{target}", source=source, target=target))

# 4. Check that the network is complete (every node connected, every setting valid), then run all 1 200 months.
system.validate()
system.simulate(len(inflows))

Running the system produces no printout. Instead, every node keeps a record of **events**: what it received,
stored, released, lost, or failed to deliver, month by month. The farm records a `DeficitRecorded` event in every
month it gets less than it needs, and each such event carries the missing volume:

In [ ]:
# @title Code: first-read
#| label: first-read
farm = system.nodes["farm"]  # look up a node by its id

# Add up the missing volume over all deficit events, then divide by the number of years
not_delivered = sum(event.deficit for event in farm.events_of_type(DeficitRecorded))
print(f"Water not delivered: {not_delivered / YEARS:.2f} Mm³ per year, on average")

From here on, `simulate_plan(inflows, plan)` does exactly these steps (build, validate, simulate, read the
events) and hands back the results in a tidy form: the storage month by month, the shortage month by month, and
the three scores introduced next.

# First run: deliver in full until empty

In [ ]:
# @title Code: fig-first-run
#| label: fig-first-run
#| fig-cap: "No rationing. Left: water in the reservoir over twenty of the hundred years. Right: for each year, how short the worst month was."
no_rationing = simulate_plan(inflows, NO_RATIONING)  # NO_RATIONING is Plan(trigger=0, ration=1)
first_year = drought_window_start(no_rationing)  # a 20-year window that contains the worst drought

# One row per plan; the right panel shows the 20 years starting in first_year, shaded in the left panel
_ = plot_runs({"Deliver in full until empty": no_rationing}, first_year=first_year)

The left panel shows all 100 years. For each year it draws one bar: the worst month of that year, as the share of
that month's need the farm did not get. A bar of 72 % means that in its worst month the farm received only 28 %
of what it asked for; most years have no bar at all. The right panel zooms into the shaded 20 years around the
worst drought: the reservoir fills every winter (the rising lines) and is drawn down every summer, and in the
driest years it reaches zero.

We score a plan with three numbers:

- **Average shortage**: the water the farm needed but did not get, in Mm³ per year.
- **Worst month**: the share of its need the farm did not get in the single worst month of the hundred years.
- **Years short**: the number of years with at least one short month.

In [ ]:
# @title Code: first-scores
#| label: first-scores
#| echo: false
s = no_rationing.scores
say(
    f"Without rationing the farm is short in **{s.years_short} of 100 years**. On average **{s.mean_shortage:.1f} Mm³ "
    f"per year** is not delivered, which is little. But when it goes wrong, it goes badly wrong: in the worst month "
    f"the farm gets **{100 - s.worst_month:.0f} %** of what it needs ({s.worst_month:.0f} % short).",
    f"Cheklashsiz xoʻjalik **100 yildan {s.years_short} tasida** taqchillik koʻradi. Oʻrtacha **yiliga "
    f"{s.mean_shortage:.1f} mln m³** yetkazib berilmaydi, bu kam. Lekin ishkal chiqqanda, qattiq chiqadi: eng yomon "
    f"oyda xoʻjalik keragining **{100 - s.worst_month:.0f} %** ini oladi ({s.worst_month:.0f} % taqchillik).",
    f"Без лимитирования хозяйство испытывает дефицит в **{s.years_short} из 100 лет**. В среднем не подаётся "
    f"**{s.mean_shortage:.1f} млн м³ в год** — немного. Но когда дело идёт плохо, оно идёт очень плохо: в худшем "
    f"месяце хозяйство получает **{100 - s.worst_month:.0f} %** потребности (дефицит {s.worst_month:.0f} %).",
)

# Second run: ration

Two rationing plans. Both start rationing when the reservoir holds less than 16 Mm³. One cuts deliveries a little
(to 80 %), the other cuts deep (to 50 %).

In [ ]:
# @title Code: fig-second-run
#| label: fig-second-run
#| fig-cap: "Three plans, same river. The grey line marks the trigger."
# A Plan holds the two knobs. Below 16 Mm³ in store, deliver 80 % (lightly) or 50 % (deep) of the summer need.
ration_lightly = simulate_plan(inflows, Plan(trigger=16, ration=0.8))
cut_deep = simulate_plan(inflows, Plan(trigger=16, ration=0.5))

# A dictionary of name -> run; the names become the panel titles
runs = {
    "Deliver in full until empty": no_rationing,
    "Ration lightly (below 16 Mm³, deliver 80 %)": ration_lightly,
    "Cut deep (below 16 Mm³, deliver 50 %)": cut_deep,
}
_ = plot_runs(runs, first_year=first_year)

Look at the bars on the left. "Ration lightly" turns the few tall bars into many short ones of 20 %, but the
tallest bar in the drought stays. Under "Cut deep" no bar is taller than 50 %: the ration itself is the worst that
happens, because the reservoir never runs dry. The price shows in the other two scores.

In [ ]:
# @title Code: tbl-three-plans
#| label: tbl-three-plans
#| tbl-cap: "The three plans, scored."
#| echo: false
pd.DataFrame(
    [
        (name, run.scores.mean_shortage, run.scores.worst_month, run.scores.years_short)
        for name, run in runs.items()
    ],
    columns=["Plan", "Average shortage (Mm³/yr)", "Worst month (% short)", "Years short (of 100)"],
).style.hide(axis="index").format({"Average shortage (Mm³/yr)": "{:.1f}", "Worst month (% short)": "{:.0f}"})

In [ ]:
# @title Code: second-scores
#| label: second-scores
#| echo: false
light, deep = ration_lightly.scores, cut_deep.scores
deep_min = cut_deep.storage.min()  # the lowest the reservoir ever gets under the deep cut
say(
    f"- **How to read the storage panel.** The grey line is the trigger, 16 Mm³. Every summer month in which the "
    f"blue line is below it is a rationed month. Without rationing the line hits zero in the drought; under "
    f"\"Cut deep\" it never falls below {deep_min:.1f} Mm³, because halving the deliveries keeps water back.\n"
    f"- **How to read the bars.** Under \"Ration lightly\" most bars are 20 % tall: that is the ration itself, "
    f"in every year the trigger is reached. There are more of them ({light.years_short} years instead of "
    f"{s.years_short}), and the tallest one, {light.worst_month:.0f} %, is still there. Under \"Cut deep\" the bars "
    f"are 50 % at most and there are {deep.years_short} of them.\n"
    f"- **Rationing lightly does not help here.** The worst month is still {light.worst_month:.0f} % short: in the "
    f"worst drought the reservoir runs dry anyway. The farm is simply short more often "
    f"({light.years_short} years instead of {s.years_short}).\n"
    f"- **Cutting deep works, at a price.** The worst month improves from {s.worst_month:.0f} % to "
    f"{deep.worst_month:.0f} % short. But {deep.mean_shortage:.1f} Mm³ per year is not delivered instead of "
    f"{s.mean_shortage:.1f}, and the farm is short in {deep.years_short} years instead of {s.years_short}.\n\n"
    f"So there is no free lunch: a milder worst month costs water. How much, exactly?",
    f"- **Suv hajmi panelini qanday oʻqish.** Kulrang chiziq — chegara, 16 mln m³. Koʻk chiziq undan pastda boʻlgan "
    f"har bir yoz oyi — cheklangan oy. Cheklashsiz chiziq qurgʻoqchilikda nolga tushadi; \"Keskin kesish\" ostida "
    f"u hech qachon {deep_min:.1f} mln m³ dan pastga tushmaydi, chunki suv berishni yarmiga kamaytirish suvni saqlab "
    f"qoladi.\n"
    f"- **Ustunlarni qanday oʻqish.** \"Ozgina cheklash\" ostida koʻp ustunlar 20 % balandlikda: bu cheklashning "
    f"oʻzi, chegaraga yetilgan har bir yilda. Ular koʻproq ({s.years_short} oʻrniga {light.years_short} yil), eng "
    f"balandi esa, {light.worst_month:.0f} %, hali ham turibdi. \"Keskin kesish\" ostida ustunlar koʻpi bilan 50 % va "
    f"ular {deep.years_short} ta.\n"
    f"- **Ozgina cheklash bu yerda yordam bermaydi.** Eng yomon oy hali ham {light.worst_month:.0f} % taqchil: eng "
    f"qattiq qurgʻoqchilikda ombor baribir quriydi. Xoʻjalik shunchaki koʻproq taqchillik koʻradi "
    f"({s.years_short} oʻrniga {light.years_short} yil).\n"
    f"- **Keskin kesish ishlaydi, lekin bahosi bor.** Eng yomon oy {s.worst_month:.0f} % dan {deep.worst_month:.0f} % "
    f"taqchillikka yaxshilanadi. Lekin {s.mean_shortage:.1f} oʻrniga yiliga {deep.mean_shortage:.1f} mln m³ "
    f"yetkazilmaydi, va xoʻjalik {s.years_short} oʻrniga {deep.years_short} yilda taqchillik koʻradi.\n\n"
    f"Demak, tekin tushlik yoʻq: yumshoqroq eng yomon oy suvga tushadi. Aniq qanchaga?",
    f"- **Как читать панель с запасом.** Серая линия — порог, 16 млн м³. Каждый летний месяц, в который синяя "
    f"линия ниже порога, — месяц с лимитированием. Без лимитирования линия в засуху доходит до нуля; при "
    f"«сильном урезании» она никогда не опускается ниже {deep_min:.1f} млн м³, потому что половинная подача "
    f"сберегает воду.\n"
    f"- **Как читать столбики.** При «лёгком лимитировании» большинство столбиков высотой 20 %: это само "
    f"урезание, в каждый год, когда достигнут порог. Их больше ({light.years_short} лет вместо {s.years_short}), а "
    f"самый высокий, {light.worst_month:.0f} %, никуда не делся. При «сильном урезании» столбики не выше 50 %, и их "
    f"{deep.years_short}.\n"
    f"- **Лёгкое лимитирование здесь не помогает.** Худший месяц по-прежнему с дефицитом {light.worst_month:.0f} %: "
    f"в самую сильную засуху водохранилище всё равно пустеет. Хозяйство просто чаще испытывает дефицит "
    f"({light.years_short} лет вместо {s.years_short}).\n"
    f"- **Сильное урезание работает, но имеет цену.** Худший месяц улучшается с {s.worst_month:.0f} % до "
    f"{deep.worst_month:.0f} % дефицита. Но не подаётся {deep.mean_shortage:.1f} млн м³ в год вместо "
    f"{s.mean_shortage:.1f}, и хозяйство испытывает дефицит в {deep.years_short} годах вместо {s.years_short}.\n\n"
    f"Бесплатных обедов нет: более мягкий худший месяц стоит воды. Сколько именно?",
)

## Try your own plan

In [ ]:
# @title Code: fig-my-plan
#| label: fig-my-plan
#| fig-cap: "Your plan against no rationing."
my_plan = Plan(trigger=22, ration=0.55)  # change these two numbers and run the cell again

mine = simulate_plan(inflows, my_plan)
_ = plot_runs({"Deliver in full until empty": no_rationing, "My plan": mine}, first_year=first_year)
print(mine.scores)  # the three scores of your plan

The trigger can be anything from 0 to 40 Mm³ (the capacity), the ration anything from 0 to 1. A trigger of 40
means "always ration in summer".

What to look for: the two rows share their axes, so compare them directly. Your plan's three scores are printed
under the figure. A plan is better only if it improves one score without worsening the other two; most changes
trade one for another. Two useful extremes: trigger 0 is "never ration" (the first row again), trigger 40 is
"ration in every summer month".

# Every plan at once

> **The problem in formulas.** Months are numbered $t = 1, \dots, 12N$ over $N = 100$ years; $m(t)$ is the month
> within its year and $S$ the six summer months. At the start of month $t$ the reservoir holds $s_t$ and receives
> the inflow $q_t$; what exceeds the capacity $K = 40$ Mm³ spills, and the loss $\ell_t$ (zero until the
> evaporation section) is taken. The water the rule sees is $\hat s_t$. A plan is the pair $\theta = (\tau, \rho)$,
> trigger and ration, and the farm needs $d_m$ (0.83 Mm³ in a winter month, 7.5 in a summer month). The release
> and the water balance are
>
$$
r_t(\theta) = \min\{\hat s_t,\; \rho_t\, d_{m(t)}\}, \qquad
\rho_t = \begin{cases} \rho & \text{if } m(t) \in S \text{ and } \hat s_t < \tau \\ 1 & \text{otherwise,} \end{cases}
$$
>
$$
\hat s_t = \min\{K,\; s_t + q_t\} - \ell_t, \qquad s_{t+1} = \hat s_t - r_t,
$$
>
> and the shortage of month $t$ is $\delta_t = d_{m(t)} - r_t$. The two scores, in Mm³ per year and in percent, are
>
$$
f_1(\theta) = \frac{1}{N}\sum_{t=1}^{12N} \delta_t(\theta), \qquad
f_2(\theta) = 100 \max_t \frac{\delta_t(\theta)}{d_{m(t)}},
$$
>
> and the problem is to make both small at once:
>
$$
\min_{\theta \in [0,K] \times [0,1]} \big(f_1(\theta),\, f_2(\theta)\big),
$$
>
> Two numbers cannot both be smallest at once, so "solve" means: find the plans no other plan beats. A plan
> $\theta$ is unbeaten if there is no $\theta'$ with $f_1(\theta') \le f_1(\theta)$ and $f_2(\theta') \le f_2(\theta)$,
> one of them strictly; the set of unbeaten plans is the Pareto front. The grid evaluates $f$ at 21 × 21 values
> of $\theta$; the optimizer searches the whole rectangle.

Two knobs are few enough to simply try everything: 21 triggers × 21 rations = 441 plans, 441 TaqSim runs.

In [ ]:
# @title Code: grid
#| label: grid
plans = plan_grid()  # 441 Plans: triggers 0, 2, 4, ... 40 Mm³ combined with rations 0, 0.05, 0.10, ... 1
table = score_plans(inflows, plans)  # runs each plan through TaqSim; one row per plan with its three scores

table.sample(5, random_state=1).sort_index().style.hide(axis="index").format(precision=2)  # five rows, as a taste

Each row is one plan: its two knobs, its three scores, and an `unbeaten` flag explained below. All 441 runs
take a few seconds.

The next figure shows all 441 plans on three maps, one per score. Each map is a chessboard: trigger across, ration
up, one square per plan, darker is worse. Left map: the further right and down (ration earlier, deliver less), the
more water is held back and never delivered. Middle map: the lightest squares, the mildest worst months, sit at the
top right, a high trigger with a ration around 60 to 75 %; the dark band at the bottom is "deliver almost nothing
once rationing starts". Right map: the number of short years grows with the trigger, because rationing is switched
on in more years. No square is light on all three maps. That is the trade-off.

In [ ]:
# @title Code: fig-maps
#| label: fig-maps
#| fig-cap: "Each score for every plan. Darker is worse. The three maps do not agree on where the good plans are."
#| echo: false
_ = plot_score_maps(table)

Now put two of the scores against each other. Every plan becomes one dot.

In [ ]:
# @title Code: fig-tradeoff
#| label: fig-tradeoff
#| fig-cap: "All 441 plans. Left is better (less water undelivered). Down is better (milder worst month). A plan is 'beaten' if another plan is at least as good on both counts and better on one."
# Every row of the table becomes a dot; `named` runs are circled and labelled
_ = plot_tradeoff(table, named={"deliver in full": no_rationing, "ration lightly": ration_lightly, "cut deep": cut_deep})

The black dots are the **unbeaten plans**. For each of them, the only way to get a milder worst month is to leave
more water undelivered. They are the only plans worth discussing; which one to pick is a judgement, not a
calculation.

In [ ]:
# @title Code: tradeoff-howto
#| label: tradeoff-howto
#| echo: false
front = front_of(table)
head, tail = front.iloc[0], front.iloc[-1]
say(
    f"How to read the chart: one dot per plan. Across, the average shortage, less is better; up, the worst month, "
    f"lower is better. The best corner is the bottom left, and no plan reaches it. The circled dots are the three "
    f"plans of the earlier sections: \"ration lightly\" sits almost on top of \"deliver in full\", \"cut deep\" is "
    f"on the staircase. The staircase joins the unbeaten plans. \"Plans lie further right\" means the axis is cut "
    f"off so that the interesting part stays readable.\n\n"
    f"The table below lists the staircase from its top-left end to its bottom-right end: from no rationing "
    f"({head['mean_shortage']:.2f} Mm³ per year undelivered, worst month {head['worst_month']:.0f} % short, "
    f"{head['years_short']:.0f} years short) to the deepest plan still worth having (trigger {tail['trigger']:.0f} Mm³, "
    f"ration {tail['ration']:.2f}: {tail['mean_shortage']:.2f} Mm³ per year, {tail['worst_month']:.0f} %, "
    f"{tail['years_short']:.0f} years short). Every step down the table buys a milder worst month with more water "
    f"left undelivered and more years with some shortage.",
    f"Diagrammani qanday oʻqish: har bir nuqta bitta reja. Koʻndalangiga — oʻrtacha taqchillik, kami yaxshi; yuqoriga "
    f"— eng yomon oy, pasti yaxshi. Eng yaxshi burchak — pastki chap, va hech bir reja unga yetmaydi. Doira ichidagi "
    f"nuqtalar — oldingi boʻlimlardagi uchta reja: \"ozgina cheklash\" deyarli \"toʻliq berish\" ustida turibdi, "
    f"\"keskin kesish\" zinapoyada. Zinapoya yengilmagan rejalarni birlashtiradi. \"Rejalar oʻngroqda\" degani — "
    f"qiziqarli qism oʻqilishi uchun oʻq kesilgan.\n\n"
    f"Quyidagi jadval zinapoyani yuqori chap uchidan pastki oʻng uchigacha sanab oʻtadi: cheklashsizdan (yiliga "
    f"{head['mean_shortage']:.2f} mln m³ yetkazilmagan, eng yomon oy {head['worst_month']:.0f} % taqchil, "
    f"{head['years_short']:.0f} taqchil yil) hali ham arziydigan eng chuqur rejagacha (chegara {tail['trigger']:.0f} "
    f"mln m³, cheklash {tail['ration']:.2f}: yiliga {tail['mean_shortage']:.2f} mln m³, {tail['worst_month']:.0f} %, "
    f"{tail['years_short']:.0f} taqchil yil). Jadvalda har bir qadam pastga yumshoqroq eng yomon oyni koʻproq "
    f"yetkazilmagan suv va koʻproq taqchil yil evaziga sotib oladi.",
    f"Как читать график: одна точка — один план. По горизонтали средний дефицит, меньше — лучше; по вертикали худший "
    f"месяц, ниже — лучше. Лучший угол — левый нижний, и ни один план до него не дотягивает. Обведённые точки — три "
    f"плана из предыдущих разделов: «лёгкое лимитирование» лежит почти поверх «подавать полностью», «сильное урезание» "
    f"— на лестнице. Лестница соединяет непревзойдённые планы. «Планы лежат правее» означает, что ось обрезана, чтобы "
    f"интересная часть оставалась читаемой.\n\n"
    f"Таблица ниже перечисляет лестницу от её левого верхнего конца до правого нижнего: от отсутствия лимитирования "
    f"({head['mean_shortage']:.2f} млн м³ в год не подано, худший месяц {head['worst_month']:.0f} % дефицита, "
    f"{head['years_short']:.0f} дефицитных лет) до самого глубокого плана, который ещё имеет смысл (порог "
    f"{tail['trigger']:.0f} млн м³, подача {tail['ration']:.2f}: {tail['mean_shortage']:.2f} млн м³ в год, "
    f"{tail['worst_month']:.0f} %, {tail['years_short']:.0f} дефицитных лет). Каждый шаг вниз по таблице покупает "
    f"более мягкий худший месяц ценой большего объёма неподанной воды и большего числа лет с дефицитом.",
)

In [ ]:
# @title Code: tbl-front
#| label: tbl-front
#| tbl-cap: "The unbeaten plans, from least water undelivered to mildest worst month."
#| echo: false
front = front_of(table)
front[["trigger", "ration", "mean_shortage", "worst_month", "years_short"]].rename(
    columns={
        "trigger": "Trigger (Mm³)",
        "ration": "Ration",
        "mean_shortage": "Average shortage (Mm³/yr)",
        "worst_month": "Worst month (% short)",
        "years_short": "Years short (of 100)",
    }
).style.hide(axis="index").format(
    {"Trigger (Mm³)": "{:.0f}", "Ration": "{:.2f}", "Average shortage (Mm³/yr)": "{:.2f}", "Worst month (% short)": "{:.0f}"}
)

# Let the optimizer search

Trying everything worked because there are only two knobs. A real reservoir rule has dozens (different triggers and
rations for each month, several reservoirs, several users). With ten knobs and 21 values each, "everything" is
more runs than anyone can wait for.

The optimizer (TaqSim uses a method called NSGA-II) searches instead. It starts with random plans, keeps the
unbeaten ones, makes new plans by mixing and nudging them, and repeats. We give it the system and the two goals:

In [ ]:
# @title Code: objectives-source
#| label: objectives-source
#| echo: false
show_source(objectives)

Each `Objective` has a name, a direction, and a function that turns a simulated system into one number.

In [ ]:
# @title Code: optimise
#| label: optimise
found = optimise_plans(
    inflows,
    pop_size=40,  # the optimizer works with 40 plans at a time
    generations=40,  # and improves them 40 times: 1 600 TaqSim runs in all
    seed=42,  # the starting plans are random; a fixed seed makes the result reproducible
)
print(f"The optimizer returned {len(found)} plans, {len(front_of(found))} of them different.")

Inside, `optimise_plans` is one call to TaqSim, `optimize(system, objectives, timesteps, pop_size, generations,
seed)`, followed by a re-run of each returned plan to read off its scores. The result table has the same columns
as the grid table above.

In [ ]:
# @title Code: fig-optimizer
#| label: fig-optimizer
#| fig-cap: "The optimizer's plans (blue) on top of the 441 grid plans. It finds the same curve and fills the gaps, because it is not tied to the grid's steps."
_ = plot_tradeoff(table, found=found)

In [ ]:
# @title Code: optimizer-howto
#| label: optimizer-howto
#| echo: false
distinct = len(front_of(found))
grid_best = front["worst_month"].min()
say(
    f"How to read it: the blue dots are the {len(found)} plans the optimizer finished with, {distinct} of them "
    f"different. Most sit on or just beside the black staircase, and several lie between its steps: the optimizer may "
    f"choose any trigger and any ration, not only the grid's steps of 2 Mm³ and 5 %. A few blue dots reach further "
    f"right than the last black one: the optimizer also found plans with a worst month below {grid_best:.0f} %, at a "
    f"high price in water. It did this with 1 600 runs and without ever seeing the grid.",
    f"Buni qanday oʻqish: koʻk nuqtalar — optimallashtiruvchi yakunlagan {len(found)} reja, ulardan {distinct} tasi "
    f"har xil. Koʻpchiligi qora zinapoyaning ustida yoki yonida, bir nechtasi esa uning pogʻonalari orasida: "
    f"optimallashtiruvchi istalgan chegara va istalgan cheklashni tanlashi mumkin, toʻrning 2 mln m³ va 5 % lik "
    f"qadamlarini emas. Bir nechta koʻk nuqta oxirgi qora nuqtadan oʻngroqqa yetadi: optimallashtiruvchi eng yomon "
    f"oyi {grid_best:.0f} % dan past rejalarni ham topdi, suv boʻyicha qimmat narxda. U buni 1 600 ta hisob bilan va "
    f"toʻrni umuman koʻrmasdan qildi.",
    f"Как это читать: синие точки — {len(found)} планов, с которыми оптимизатор закончил, {distinct} из них разные. "
    f"Большинство лежат на чёрной лестнице или рядом с ней, а несколько — между её ступенями: оптимизатор может "
    f"выбрать любой порог и любую долю подачи, а не только шаги сетки в 2 млн м³ и 5 %. Несколько синих точек "
    f"уходят правее последней чёрной: оптимизатор нашёл и планы с худшим месяцем ниже {grid_best:.0f} %, по высокой "
    f"цене в воде. Он сделал это за 1 600 прогонов, ни разу не увидев сетку.",
)

# What to notice

In [ ]:
# @title Code: notice
#| label: notice
#| echo: false
first, last = front.iloc[0], front.iloc[-1]
knee = front.loc[(front["worst_month"] - 50.0).abs().idxmin()]
beaten = int((~table["unbeaten"]).sum())
best_volume = table.loc[table["mean_shortage"].idxmin()]
say(
    f"**1. What you count decides the answer.** If the only goal is to deliver as much water as possible, there is "
    f"one best plan and it is *no rationing* (average shortage {best_volume['mean_shortage']:.2f} Mm³ per year, the "
    f"lowest of all 441 plans). The trade-off only appears once you also care about the worst month. Choosing the "
    f"goals is the most important modelling decision, and it is not a technical one.\n\n"
    f"**2. The curve bends.** Going from {first['worst_month']:.0f} % to {knee['worst_month']:.0f} % short in the "
    f"worst month costs {knee['mean_shortage'] - first['mean_shortage']:.1f} Mm³ per year. Going on from "
    f"{knee['worst_month']:.0f} % to {last['worst_month']:.0f} % costs another "
    f"{last['mean_shortage'] - knee['mean_shortage']:.1f} Mm³ per year. The first steps are cheap, the last ones "
    f"very expensive. A plan near the bend is the natural place to start the discussion.\n\n"
    f"**3. Being short more often is part of the price.** Along the curve the number of short years rises from "
    f"{first['years_short']:.0f} to {last['years_short']:.0f} of 100.\n\n"
    f"**4. Most plans are simply bad.** {beaten} of the 441 plans are beaten, including the reasonable-sounding "
    f"\"ration lightly\". Intuition is a poor guide here; the search is not.",
    f"**1. Nimani hisoblashingiz javobni belgilaydi.** Agar yagona maqsad iloji boricha koʻp suv yetkazish boʻlsa, "
    f"bitta eng yaxshi reja bor va bu — *cheklash yoʻq* (oʻrtacha taqchillik yiliga "
    f"{best_volume['mean_shortage']:.2f} mln m³, barcha 441 reja ichida eng kami). Murosa faqat eng yomon oy ham "
    f"sizga muhim boʻlgandagina paydo boʻladi. Maqsadlarni tanlash — modellashtirishdagi eng muhim qaror, va u "
    f"texnik qaror emas.\n\n"
    f"**2. Egri chiziq bukiladi.** Eng yomon oyda {first['worst_month']:.0f} % dan {knee['worst_month']:.0f} % "
    f"taqchillikka oʻtish yiliga {knee['mean_shortage'] - first['mean_shortage']:.1f} mln m³ ga tushadi. "
    f"{knee['worst_month']:.0f} % dan {last['worst_month']:.0f} % gacha davom etish yana yiliga "
    f"{last['mean_shortage'] - knee['mean_shortage']:.1f} mln m³ ga tushadi. Birinchi qadamlar arzon, oxirgilari "
    f"juda qimmat. Muhokamani bukilish yaqinidagi rejadan boshlash tabiiy.\n\n"
    f"**3. Koʻproq taqchillik koʻrish — bahoning bir qismi.** Egri chiziq boʻylab taqchil yillar soni 100 dan "
    f"{first['years_short']:.0f} tadan {last['years_short']:.0f} tagacha oshadi.\n\n"
    f"**4. Koʻpchilik rejalar shunchaki yomon.** 441 rejadan {beaten} tasi ustun kelingan, shu jumladan oqilona "
    f"tuyulgan \"ozgina cheklash\" ham. Bu yerda sezgi yomon yoʻlboshchi; qidiruv esa yaxshi.",
    f"**1. Ответ определяет то, что вы считаете.** Если единственная цель — подать как можно больше воды, есть один "
    f"лучший план, и это *без лимитирования* (средний дефицит {best_volume['mean_shortage']:.2f} млн м³ в год, "
    f"наименьший из всех 441 планов). Компромисс появляется только тогда, когда вам важен и худший месяц. Выбор "
    f"целей — самое важное решение при моделировании, и оно не техническое.\n\n"
    f"**2. Кривая изгибается.** Переход от {first['worst_month']:.0f} % к {knee['worst_month']:.0f} % дефицита в "
    f"худшем месяце стоит {knee['mean_shortage'] - first['mean_shortage']:.1f} млн м³ в год. Дальнейший переход от "
    f"{knee['worst_month']:.0f} % к {last['worst_month']:.0f} % стоит ещё "
    f"{last['mean_shortage'] - knee['mean_shortage']:.1f} млн м³ в год. Первые шаги дёшевы, последние очень дороги. "
    f"План возле изгиба — естественная отправная точка для обсуждения.\n\n"
    f"**3. Более частый дефицит — часть цены.** Вдоль кривой число дефицитных лет растёт с "
    f"{first['years_short']:.0f} до {last['years_short']:.0f} из 100.\n\n"
    f"**4. Большинство планов просто плохи.** {beaten} из 441 планов превзойдены, включая разумное на слух «лёгкое "
    f"лимитирование». Интуиция здесь плохой советчик; поиск — хороший.",
)

# One more block: evaporation

So far, water saved in the reservoir waits there for free. In a hot climate it does not: some of it evaporates. In
TaqSim that is one more small class, a *loss rule*, plugged into the reservoir:

In [ ]:
# @title Code: evaporation-source
#| label: evaporation-source
#| echo: false
show_source(SummerEvaporation)

`calculate(...)` is called by TaqSim once per month, after the inflow has been stored and before the release. It
returns what was lost and why; TaqSim subtracts it from the storage and records a `WaterLost` event.

In [ ]:
# @title Code: fig-evaporation
#| label: fig-evaporation
#| fig-cap: "The unbeaten plans without and with evaporation. With evaporation every plan delivers less, and a mild worst month becomes much more expensive."
# Same 441 plans, same river, but now the reservoir loses 3 % of its content in each summer month
with_evaporation = score_plans(inflows, plan_grid(), evaporation_share=EVAPORATION_SHARE_EXAMPLE)

# Draw the unbeaten plans of both tables on one chart
_ = plot_front_shift({"no evaporation": table, "3 % of the stored water evaporates each summer month": with_evaporation})

In [ ]:
# @title Code: evaporation-text
#| label: evaporation-text
#| echo: false
front_e = front_of(with_evaporation)
at_50 = front[front["worst_month"] <= 50.0].iloc[0]
at_50_e = front_e[front_e["worst_month"] <= 50.0].iloc[0]
say(
    f"The two staircases are the unbeaten plans without (blue) and with (orange) evaporation. The orange one lies to "
    f"the right of and above the blue one: every plan delivers less, and the same worst month costs more water. It "
    f"also stops higher: with evaporation no plan gets the worst month below {front_e['worst_month'].min():.0f} %, "
    f"against {front['worst_month'].min():.0f} % without. "
    f"The 3 % is an illustrative figure, not a measurement. The effect is large: holding the worst month to 50 % "
    f"short costs {at_50['mean_shortage']:.1f} Mm³ per year without evaporation and "
    f"{at_50_e['mean_shortage']:.1f} Mm³ per year with it. Saving water for later is worth less when the saved "
    f"water does not keep.",
    f"Ikkita zinapoya — bugʻlanishsiz (koʻk) va bugʻlanish bilan (toʻq sariq) yengilmagan rejalar. Toʻq sariq "
    f"zinapoya koʻkning oʻngida va yuqorisida: har bir reja kamroq yetkazadi, va bir xil eng yomon oy koʻproq suvga "
    f"tushadi. U yuqoriroqda ham toʻxtaydi: bugʻlanish bilan hech bir reja eng yomon oyni "
    f"{front_e['worst_month'].min():.0f} % dan pastga tushira olmaydi, bugʻlanishsiz esa "
    f"{front['worst_month'].min():.0f} % gacha. "
    f"3 % — oʻlchov emas, tushuntirish uchun olingan raqam. Taʼsiri katta: eng yomon oyni 50 % taqchillikda "
    f"ushlab turish bugʻlanishsiz yiliga {at_50['mean_shortage']:.1f} mln m³, bugʻlanish bilan esa yiliga "
    f"{at_50_e['mean_shortage']:.1f} mln m³ ga tushadi. Saqlangan suv saqlanmasa, uni keyinga asrash kamroq "
    f"qadrli.",
    f"Две лестницы — непревзойдённые планы без испарения (синяя) и с испарением (оранжевая). Оранжевая лежит правее "
    f"и выше синей: каждый план подаёт меньше, а тот же худший месяц стоит больше воды. Она и обрывается выше: с "
    f"испарением ни один план не опускает худший месяц ниже {front_e['worst_month'].min():.0f} %, без испарения — "
    f"до {front['worst_month'].min():.0f} %. "
    f"3 % — иллюстративное число, не измерение. Эффект велик: удержать худший месяц на уровне 50 % дефицита стоит "
    f"{at_50['mean_shortage']:.1f} млн м³ в год без испарения и {at_50_e['mean_shortage']:.1f} млн м³ в год с "
    f"ним. Беречь воду на потом менее выгодно, когда сбережённая вода не сохраняется.",
)

# One more user: the winter fields

The reservoir fills in winter, and in the Zarafshan oasis winter is not idle. Winter wheat is watered, and before
the spring sowing the saline fields are leached: flooded, so that the salt is washed down below the roots. Both
take water from the same reservoir in the months when it should be filling. We add that user as a fifth block, on
the same canal below the farm: the **winter fields**. They ask for 2.5 Mm³ in each winter month, 15 Mm³ a winter,
and nothing in summer. The number is illustrative, like the evaporation share.

In [ ]:
# @title Code: fig-winter-network
#| label: fig-winter-network
#| fig-cap: "Five blocks. The winter fields sit on the canal below the farm: in winter they take what the reservoir releases for them, and whatever the reservoir spills."
_ = plot_network(winter=True)

Every winter the operator faces a new question. Water given to the winter fields is gone before anyone knows how
big the summer melt will be. Give it in full, and a weak melt leaves the farm short in July. Hold it back, and in
most winters the water was not needed. The rule gets two more knobs, for winter:

- **Winter trigger** (Mm³): when the *water in sight* is below this, the winter fields are rationed. For now the
  water in sight is simply the storage.
- **Winter ration** (0 to 1): the share of their need the winter fields then get.

The summer knobs stay as they were. For the rest of this notebook we fix them at "cut deep" (trigger 16 Mm³,
ration 0.5), the plan from the second run, so that only the winter decision changes.

In [ ]:
# @title Code: winter-rule-source
#| label: winter-rule-source
#| echo: false
show_source(WinterRelease)

In [ ]:
# @title Code: winter-two-runs
#| label: winter-two-runs
# Summer "cut deep" in both; the winter fields served in full, or given nothing when the store is below 24 Mm³
winter_in_full = simulate_winter(inflows, WINTER_IN_FULL)
hold_back = simulate_winter(inflows, WinterPlan(trigger=24.0, ration=0.0))
for name, run in {"Winter fields served in full": winter_in_full, "Hold back below 24 Mm³": hold_back}.items():
    print(
        f"{name:30s} farm short {run.scores.summer_shortage:.1f} Mm³/yr, "
        f"winter fields short {run.scores.winter_shortage:.1f} Mm³/yr"
    )

In [ ]:
# @title Code: winter-two-runs-text
#| label: winter-two-runs-text
#| echo: false
full, back = winter_in_full.scores, hold_back.scores
say(
    f"Served in full, the winter fields get almost all of their 15 Mm³, and the farm's shortage rises from "
    f"{cut_deep.scores.mean_shortage:.1f} Mm³ a year (the same summer plan without the winter fields) to "
    f"{full.summer_shortage:.1f}. Holding back below 24 Mm³ gives the farm back most of that, "
    f"{full.summer_shortage - back.summer_shortage:.1f} Mm³ a year, at a price of "
    f"{back.winter_shortage - full.winter_shortage:.1f} Mm³ a year for the winter fields. Even \"nothing\" is not "
    f"quite nothing: in wet winters the reservoir spills, and the spilled water reaches the fields anyway.\n\n"
    f"The two goals are now the two users: the water the farm did not get in summer, and the water the winter "
    f"fields did not get in winter, both in Mm³ a year, both less is better. As before, we try every winter plan: "
    f"triggers from 0 to 40 Mm³ in steps of 1, rations in steps of 0.05, 861 plans.",
    f"Toʻliq berilganda qishki dalalar oʻzining 15 mln m³ ining deyarli hammasini oladi, xoʻjalikning taqchilligi "
    f"esa yiliga {cut_deep.scores.mean_shortage:.1f} mln m³ dan (qishki dalalarsiz oʻsha yozgi reja) "
    f"{full.summer_shortage:.1f} ga koʻtariladi. 24 mln m³ dan pastda ushlab turish xoʻjalikka buning koʻp qismini, "
    f"yiliga {full.summer_shortage - back.summer_shortage:.1f} mln m³ ni qaytaradi, qishki dalalar uchun esa yiliga "
    f"{back.winter_shortage - full.winter_shortage:.1f} mln m³ ga tushadi. Hatto \"hech narsa\" ham toʻliq hech "
    f"narsa emas: sersuv qishlarda ombor toshadi va toshgan suv baribir dalalarga yetib boradi.\n\n"
    f"Endi ikki maqsad — ikki isteʼmolchi: xoʻjalik yozda olmagan suv va qishki dalalar qishda olmagan suv, "
    f"ikkalasi ham yiliga mln m³ da, ikkalasida ham kam boʻlgani yaxshi. Avvalgidek, har bir qishki rejani sinab "
    f"koʻramiz: chegaralar 0 dan 40 mln m³ gacha 1 qadam bilan, ulushlar 0,05 qadam bilan, 861 ta reja.",
    f"При полной подаче зимние поля получают почти все свои 15 млн м³, а дефицит хозяйства растёт с "
    f"{cut_deep.scores.mean_shortage:.1f} млн м³ в год (тот же летний план без зимних полей) до "
    f"{full.summer_shortage:.1f}. Придерживание ниже 24 млн м³ возвращает хозяйству большую часть этого, "
    f"{full.summer_shortage - back.summer_shortage:.1f} млн м³ в год, ценой "
    f"{back.winter_shortage - full.winter_shortage:.1f} млн м³ в год для зимних полей. Даже «ничего» — не совсем "
    f"ничего: в многоводные зимы водохранилище сбрасывает излишек, и эта вода всё равно доходит до полей.\n\n"
    f"Две цели — теперь два водопользователя: вода, которую хозяйство не получило летом, и вода, которую зимние "
    f"поля не получили зимой, обе в млн м³ в год, обе — чем меньше, тем лучше. Как и раньше, перебираем все "
    f"зимние планы: пороги от 0 до 40 млн м³ с шагом 1, доли с шагом 0,05, 861 план.",
)

> **The winter problem in formulas.** The summer plan is fixed, $\theta^* = (16, 0.5)$; the winter plan is
> $\psi = (\tau_w, \rho_w)$. The winter fields need $w_m = 2.5$ Mm³ in a winter month and nothing in summer. In a
> winter month the rule compares the water in sight $v_t$ with the winter trigger: the storage alone for now, the
> storage plus the forecast $F_y$ of the coming summer in the next section. The farm is served first and the winter
> fields get the rest:
>
$$
r_t = \min\{\hat s_t,\; d_{m(t)} + \rho^{w}_t\, w_{m(t)}\}, \qquad
\rho^{w}_t = \begin{cases} \rho_w & \text{if } v_t < \tau_w \\ 1 & \text{otherwise,} \end{cases} \qquad
v_t = \hat s_t \;\text{ or }\; \hat s_t + F_{y(t)},
$$
>
> Summer months follow the summer rule above. The two scores are the two users' shortages, in Mm³ per year, and
> the problem is the same kind as before, over $\tau_w \in [0, 40]$ (or $[0, 120]$ with a forecast) and
> $\rho_w \in [0, 1]$:
>
$$
g_1(\psi) = \frac{1}{N}\sum_{t:\, m(t) \in S} \delta^{\text{farm}}_t, \qquad
g_2(\psi) = \frac{1}{N}\sum_{t:\, m(t) \notin S} \delta^{\text{fields}}_t, \qquad
\min_{\psi} \big(g_1(\psi),\, g_2(\psi)\big).
$$

In [ ]:
# @title Code: fig-winter-front
#| label: fig-winter-front
#| fig-cap: "The unbeaten winter plans when the winter decision looks at the storage alone. Left is better for the farm, down is better for the winter fields."
# 861 winter plans, summer fixed at "cut deep"; the water in sight is the storage, so triggers run to 40 Mm³
storage_only = score_winter_plans(inflows, winter_grid(n_trigger=41, trigger_max=CAPACITY_MM3))
_ = plot_winter_fronts({"winter decided on the storage alone": storage_only})

In [ ]:
# @title Code: winter-front-text
#| label: winter-front-text
#| echo: false
at_2 = least_winter_shortage(storage_only, 2.0)
at_3 = least_winter_shortage(storage_only, 3.0)
say(
    f"The same bent curve as in summer. Its bottom-right end is \"serve in full\": the winter fields lack almost "
    f"nothing, the farm is short {full.summer_shortage:.1f} Mm³ a year. Moving up and left, each step gives the "
    f"farm back some water and takes more from the winter fields. To hold the farm's shortage to 3 Mm³ a year, the "
    f"winter fields must give up {at_3:.1f} Mm³ a year; to hold it to 2, {at_2:.1f}. The operator who only "
    f"watches the storage pays this price in every winter the reservoir is low, including the many that a strong "
    f"melt would have rescued.",
    f"Yozdagi kabi bukilgan egri chiziq. Uning pastki oʻng uchi — \"toʻliq berish\": qishki dalalarga deyarli hech "
    f"narsa yetishmaydi, xoʻjalik esa yiliga {full.summer_shortage:.1f} mln m³ taqchil. Yuqoriga va chapga "
    f"yurganda har bir qadam xoʻjalikka bir oz suv qaytaradi va qishki dalalardan koʻproq oladi. Xoʻjalik "
    f"taqchilligini yiliga 3 mln m³ da ushlab turish uchun qishki dalalar yiliga {at_3:.1f} mln m³ dan voz kechishi "
    f"kerak; 2 da ushlab turish uchun — {at_2:.1f}. Faqat ombordagi suvga qaraydigan operator bu narxni ombor kam "
    f"suvli boʻlgan har bir qishda toʻlaydi, shu jumladan kuchli erish qutqarib qoladigan koʻplab qishlarda ham.",
    f"Та же изогнутая кривая, что и летом. Её нижний правый конец — «подавать полностью»: зимним полям почти "
    f"ничего не не хватает, хозяйству не хватает {full.summer_shortage:.1f} млн м³ в год. Двигаясь вверх и влево, "
    f"каждый шаг возвращает хозяйству немного воды и отнимает больше у зимних полей. Чтобы удержать дефицит "
    f"хозяйства на 3 млн м³ в год, зимние поля должны отдать {at_3:.1f} млн м³ в год; чтобы удержать на 2 — "
    f"{at_2:.1f}. Оператор, который смотрит только на запас, платит эту цену каждую зиму, когда водохранилище "
    f"маловодно, включая те многие зимы, которые сильное таяние всё равно бы спасло.",
)

# A forecast of the summer, decided in winter

The operator holds back because the summer is unknown. But it is not entirely unknown. By late winter most of the
snow that will melt in summer has fallen, and the hydrometeorological services of Central Asia forecast the
April–September flow of their rivers every spring, from the snow cover, the winter precipitation and the flow of
the past months (Apel et al., 2018). Suppose our operator has such a forecast at the start of every winter: one
number, the inflow of the coming summer. In reality it arrives late in the winter, once the snow has fallen; here
the operator gets it at the start of winter, which is the best case, and exercise 6 asks what a later forecast is
still worth.

The rule needs one change. The water in sight is no longer the storage alone but the storage **plus the
forecast**, and the winter trigger is compared with that. Nothing else changes: same knobs, same summer plan,
same river. Because the water in sight now includes a summer, the trigger runs up to 120 Mm³ instead of 40.

We start with the best case, a forecast that is always right. It tells us the most a forecast could ever be worth
here.

In [ ]:
# @title Code: fig-forecast-front
#| label: fig-forecast-front
#| fig-cap: "The unbeaten winter plans when the winter decision looks at the storage alone (orange) and at the storage plus a perfect forecast of the coming summer (blue)."
perfect = seasonal_forecast(inflows, skill=1.0)  # one number per year: the coming summer's inflow, exactly
with_perfect = score_winter_plans(inflows, winter_grid(n_trigger=41), forecast=perfect)  # 861 plans, triggers to 120
_ = plot_winter_fronts({"storage plus a perfect forecast": with_perfect, "storage alone": storage_only})

In [ ]:
# @title Code: fig-winter-decisions
#| label: fig-winter-decisions
#| fig-cap: "Two winter plans with the same summer shortage, 2 Mm³ a year at most, in the twenty years around the worst drought. Top: what the winter fields received each winter. Bottom: the summer that followed."
# From each curve, the plan that holds the farm's shortage to 2 Mm³ a year with the least loss for the winter fields
chosen = {
    "storage plus a perfect forecast": winter_plan_within(with_perfect, 2.0),
    "storage alone": winter_plan_within(storage_only, 2.0),
}
runs = {
    label: simulate_winter(inflows, winter, forecast=perfect if "forecast" in label else None)
    for label, winter in chosen.items()
}
_ = plot_winter_decisions(runs, inflows, first_year=first_year)

In [ ]:
# @title Code: forecast-front-text
#| label: forecast-front-text
#| echo: false
at_2_perfect = least_winter_shortage(with_perfect, 2.0)
at_3_perfect = least_winter_shortage(with_perfect, 3.0)
gain_2 = (at_2 - at_2_perfect) / at_2 * 100.0
say(
    f"- **The blue curve lies below the orange one.** At every summer shortage the farm accepts, the winter fields "
    f"lose less. Holding the farm to 2 Mm³ a year costs the winter fields {at_2:.1f} Mm³ a year without the "
    f"forecast and {at_2_perfect:.1f} with it, {gain_2:.0f} % less; at 3 Mm³ a year, {at_3:.1f} against "
    f"{at_3_perfect:.1f}.\n"
    f"- **Why.** Look at the two plans in the second figure. The plan without the forecast holds back whenever the "
    f"reservoir is low, which is mostly the winter *after* a dry summer. The plan with the forecast holds back in "
    f"the winters *before* a weak summer and serves in full before a strong one. It rations fewer winters, and the "
    f"right ones.\n"
    f"- **This gap is the ceiling.** No real forecast can do better than one that is always right. Whatever a "
    f"forecast service can offer is worth at most this much here: {at_2 - at_2_perfect:.1f} Mm³ a year for the "
    f"winter fields at a summer shortage of 2. In a system where more decisions are taken before the melt, the "
    f"ceiling is higher.",
    f"- **Koʻk egri chiziq toʻq sarigʻidan pastda yotadi.** Xoʻjalik rozi boʻlgan har bir yozgi taqchillikda qishki "
    f"dalalar kamroq yoʻqotadi. Xoʻjalikni yiliga 2 mln m³ da ushlab turish qishki dalalarga prognozsiz yiliga "
    f"{at_2:.1f} mln m³, prognoz bilan {at_2_perfect:.1f} ga tushadi, yaʼni {gain_2:.0f} % kam; yiliga 3 mln m³ "
    f"da — {at_3:.1f} ga qarshi {at_3_perfect:.1f}.\n"
    f"- **Nima uchun.** Ikkinchi rasmdagi ikki rejaga qarang. Prognozsiz reja ombor kam suvli boʻlganda ushlab "
    f"turadi, bu esa asosan quruq yozdan *keyingi* qish. Prognozli reja kuchsiz yozdan *oldingi* qishlarda ushlab "
    f"turadi va kuchli yozdan oldin toʻliq beradi. U kamroq qishni cheklaydi, va aynan keraklilarini.\n"
    f"- **Bu farq — shift.** Hech bir haqiqiy prognoz hamisha toʻgʻri chiqadiganidan yaxshiroq boʻla olmaydi. "
    f"Prognoz xizmati taklif qila oladigan narsa bu yerda eng koʻpi bilan shuncha turadi: yozgi taqchillik 2 "
    f"boʻlganda qishki dalalar uchun yiliga {at_2 - at_2_perfect:.1f} mln m³. Erishdan oldin koʻproq qaror "
    f"qabul qilinadigan tizimda shift balandroq.",
    f"- **Синяя кривая лежит ниже оранжевой.** При любом летнем дефиците, на который соглашается хозяйство, зимние "
    f"поля теряют меньше. Удержать хозяйство на 2 млн м³ в год стоит зимним полям {at_2:.1f} млн м³ в год без "
    f"прогноза и {at_2_perfect:.1f} с ним, на {gain_2:.0f} % меньше; при 3 млн м³ в год — {at_3:.1f} против "
    f"{at_3_perfect:.1f}.\n"
    f"- **Почему.** Посмотрите на два плана на второй рисунке. План без прогноза придерживает воду всякий раз, "
    f"когда водохранилище маловодно, а это в основном зима *после* сухого лета. План с прогнозом придерживает "
    f"воду в зимы *перед* слабым летом и подаёт полностью перед сильным. Он лимитирует меньше зим, и именно те, "
    f"что нужно.\n"
    f"- **Этот разрыв — потолок.** Ни один реальный прогноз не может быть лучше того, что всегда верен. Что бы ни "
    f"предложила прогностическая служба, здесь это стоит самое большее столько: {at_2 - at_2_perfect:.1f} млн м³ в "
    f"год для зимних полей при летнем дефиците 2. В системе, где больше решений принимается до таяния, потолок "
    f"выше.",
)

# A forecast that can be wrong

**The idea.** The last section gave the operator a forecast that was always right. No real forecast is. The
question of this section is the one a water authority actually faces: *a forecast that is sometimes wrong, is it
still worth acting on, and how much?* The everyday version is familiar. A weather forecast that is right seven
times in ten is far from perfect, yet you still take the umbrella when it says rain, and you are right to: you
act on it differently than on a perfect one, with less trust, but you do not throw it away. We want to see the
same thing for the reservoir, with numbers.

**How we build it.** We cannot buy a hundred years of real forecasts for a synthetic river, so we manufacture
forecasts of known quality. We take the summer that will actually come, blur it with random noise, and let a
sensible forecaster make the best guess it can from the blurred picture. One knob sets how much blur: we call it
the forecast's **skill**, from 0 to 1.

- Skill 1: no blur. The forecaster sees the summer exactly; this is the perfect forecast of the last section.
- Skill 0: all blur. The forecaster sees nothing useful and can do no better than say "an average summer" every
  year.
- In between: the forecaster sees something, and the forecasts follow the real summers partly. The technical
  name for skill is the correlation coefficient between forecasts and outcomes.

`seasonal_forecast` makes such forecasts. One property matters: the forecasts are honest. A forecaster who
knows little says something close to the average; one who knows a lot dares to say "a very dry summer". On
average the guess is neither too bold nor too timid, which is how a statistical forecast, fitted to past years,
behaves. The operator's rule is the one you know: it adds the forecast to the water in sight and compares the
sum with the winter trigger. Two examples of what the forecasts look like:

> **The forecast in formulas.** The summer inflow of year $y$ is $Q_y = e^{\mu + \sigma z_y}$ with $z_y$ a
> standard score (the river is log-normal). The forecaster sees the coming summer through noise $\varepsilon_y$,
> and issues the best guess given what it saw:
>
$$
\tilde z_y = \kappa\, z_y + \sqrt{1-\kappa^2}\; \varepsilon_y, \qquad
F_y = \mathbb{E}\big[Q_y \mid \tilde z_y\big] = \exp\!\Big(\mu + \sigma \kappa \tilde z_y + \tfrac{1}{2}\sigma^2 (1-\kappa^2)\Big),
$$
>
> so that the correlation between what it sees and the truth is the skill $\kappa$, the forecast is the long-run
> mean for $\kappa = 0$ and the summer itself for $\kappa = 1$. The value of a forecast is measured at a fixed
> summer shortage $\bar c$ (2 Mm³ a year below): the winter water it saves without costing the farm anything,
>
$$
V(\kappa) = g_2^{*}(\text{no forecast}) - g_2^{*}(\kappa), \qquad
g_2^{*}(\cdot) = \min\big\{\, g_2(\psi) : g_1(\psi) \le \bar c \,\big\}.
$$
>
> $V(1)$, the value of the perfect forecast, is the ceiling.

In [ ]:
# @title Code: fig-forecast-skill
#| label: fig-forecast-skill
#| fig-cap: "One hundred summers and their forecasts, for a forecast of skill 0.5 and one of skill 0.9. On the grey line the forecast was exactly right."
_ = plot_forecast_skill(inflows, skills=(0.5, 0.9))

In [ ]:
# @title Code: skill-howto
#| label: skill-howto
#| echo: false
import numpy as np

_, summers = season_totals(inflows)
average_error = {skill: float(np.mean(np.abs(np.asarray(seasonal_forecast(inflows, skill)) - summers))) for skill in (0.0, 0.5, 0.9)}
rosy = {skill: int(np.sum(np.asarray(seasonal_forecast(inflows, skill)) > summers + 10.0)) for skill in (0.5, 0.9)}
gloomy = {skill: int(np.sum(np.asarray(seasonal_forecast(inflows, skill)) < summers - 10.0)) for skill in (0.5, 0.9)}
say(
    f"- **How to read the figure.** One dot per year: across, the summer that came; up, what the forecast said. A dot "
    f"on the grey line is a forecast that was exactly right; the further from the line, the larger the error. With "
    f"skill 0.9 the dots hug the line; with skill 0.5 they form a cloud. The typical error (the average distance "
    f"from the line) is {average_error[0.9]:.0f} Mm³ at skill 0.9 and {average_error[0.5]:.0f} Mm³ at skill 0.5, "
    f"against {average_error[0.0]:.0f} Mm³ for \"an average summer\" every year.\n"
    f"- **A weak forecast stays close to the average.** Look at the skill-0.5 cloud: it is flatter than the line, "
    f"the forecasts hardly ever say 20 or 70 even when the summer turned out so. That is not timidity, it is honesty: "
    f"a forecaster who knows little should say something close to the average, because a bold forecast from weak "
    f"evidence would be wrong more often and by more. The formula above does exactly that: it pulls the guess "
    f"towards the long-run mean in proportion to what the forecaster does not know.\n"
    f"- **Two ways to be wrong.** A forecast that is too rosy says the summer will be fine, the operator gives the "
    f"winter fields their water, and the farm is short in July. One that is too gloomy makes the operator hold back "
    f"water that was not needed. With skill 0.5 the forecast was more than 10 Mm³ too rosy in {rosy[0.5]} of the "
    f"hundred years and more than 10 Mm³ too gloomy in {gloomy[0.5]}; with skill 0.9, in {rosy[0.9]} and "
    f"{gloomy[0.9]}. The two errors are not equally bad for the two users, which is why the operator cannot simply "
    f"take a weak forecast at its word.",
    f"- **Rasmni qanday oʻqish.** Har bir yil — bitta nuqta: koʻndalangiga kelgan yoz, tikka prognoz aytgani. Kulrang "
    f"chiziq ustidagi nuqta — aynan toʻgʻri chiqqan prognoz; chiziqdan qancha uzoq boʻlsa, xato shuncha katta. Sifat "
    f"0,9 da nuqtalar chiziqqa yopishib turadi; 0,5 da bulut hosil qiladi. Odatiy xato (chiziqdan oʻrtacha "
    f"masofa) sifat 0,9 da {average_error[0.9]:.0f} mln m³, 0,5 da {average_error[0.5]:.0f} mln m³, har yili "
    f"\"oʻrtacha yoz\" deyishda esa {average_error[0.0]:.0f} mln m³.\n"
    f"- **Kuchsiz prognoz oʻrtachaga yaqin turadi.** Sifat 0,5 bulutiga qarang: u chiziqdan yotiqroq, yoz 20 yoki 70 "
    f"chiqqanda ham prognoz deyarli hech qachon 20 yoki 70 demaydi. Bu qoʻrqoqlik emas, halollik: kam biladigan "
    f"prognozchi oʻrtachaga yaqin narsa aytishi kerak, chunki kuchsiz dalillardan dadil prognoz tez-tez va koʻproq "
    f"xato boʻladi. Yuqoridagi formula aynan shuni qiladi: taxminni prognozchi bilmagan narsaga mutanosib ravishda "
    f"uzoq muddatli oʻrtacha tomon tortadi.\n"
    f"- **Xato qilishning ikki yoʻli.** Haddan tashqari umidvor prognoz yoz yaxshi boʻladi deydi, operator qishki "
    f"dalalarga suvini beradi, xoʻjalik esa iyulda taqchil qoladi. Haddan tashqari tushkun prognoz operatorni kerak "
    f"boʻlmagan suvni ushlab turishga majbur qiladi. Sifat 0,5 da prognoz yuz yildan {rosy[0.5]} tasida 10 mln m³ "
    f"dan ortiq umidvor, {gloomy[0.5]} tasida 10 mln m³ dan ortiq tushkun boʻldi; sifat 0,9 da — {rosy[0.9]} va "
    f"{gloomy[0.9]} tasida. Bu ikki xato ikki isteʼmolchi uchun bir xil yomon emas, shuning uchun operator kuchsiz "
    f"prognozga shunchaki soʻzsiz ishona olmaydi.",
    f"- **Как читать рисунок.** Одна точка — один год: по горизонтали — какое лето наступило, по вертикали — что "
    f"говорил прогноз. Точка на серой линии — прогноз, который сбылся в точности; чем дальше от линии, тем больше "
    f"ошибка. При качестве 0,9 точки жмутся к линии; при 0,5 образуют облако. Типичная ошибка (среднее расстояние "
    f"от линии) — {average_error[0.9]:.0f} млн м³ при качестве 0,9 и {average_error[0.5]:.0f} млн м³ при 0,5, против "
    f"{average_error[0.0]:.0f} млн м³, если каждый год говорить «среднее лето».\n"
    f"- **Слабый прогноз держится около среднего.** Посмотрите на облако при качестве 0,5: оно положе линии, "
    f"прогноз почти никогда не говорит 20 или 70, даже когда лето оказалось таким. Это не робость, а честность: "
    f"прогнозист, который знает мало, должен говорить что-то близкое к среднему, потому что смелый прогноз по слабым "
    f"данным ошибался бы чаще и сильнее. Формула выше делает ровно это: она притягивает оценку к многолетнему "
    f"среднему пропорционально тому, чего прогнозист не знает.\n"
    f"- **Два способа ошибиться.** Слишком радужный прогноз обещает хорошее лето, оператор отдаёт зимним полям их "
    f"воду, и хозяйству не хватает воды в июле. Слишком мрачный заставляет оператора придержать воду, которая была "
    f"не нужна. При качестве 0,5 прогноз был более чем на 10 млн м³ радужнее истины в {rosy[0.5]} из ста лет и более "
    f"чем на 10 млн м³ мрачнее — в {gloomy[0.5]}; при качестве 0,9 — в {rosy[0.9]} и {gloomy[0.9]}. Эти две ошибки "
    f"не одинаково плохи для двух водопользователей, и потому оператор не может просто верить слабому прогнозу на "
    f"слово.",
)

Now the operator uses these forecasts. The rule is unchanged: it takes the forecast at its word and adds it to
the water in sight. What changes is the plan: for each skill we try the 861 winter plans again, so that the
trigger and the ration are chosen knowing how good the forecast is.

In [ ]:
# @title Code: fig-skill-fronts
#| label: fig-skill-fronts
#| fig-cap: "The unbeaten winter plans for forecasts of growing skill. The curves move down as the skill grows, from storage alone to the perfect forecast."
# Six more forecasts of growing skill, 861 winter plans each (this cell runs for a few minutes)
tables = {1.0: with_perfect, **score_by_skill(inflows, (0.0, 0.3, 0.5, 0.65, 0.8, 0.9), winter_grid(n_trigger=41))}
_ = plot_winter_fronts(
    {
        "perfect forecast": with_perfect,
        "forecast with skill 0.8": tables[0.8],
        "forecast with skill 0.5": tables[0.5],
        "storage alone": storage_only,
    }
)

In [ ]:
# @title Code: skill-fronts-text
#| label: skill-fronts-text
#| echo: false
cases = {
    "storage alone": (storage_only, None),
    "skill 0.5": (tables[0.5], seasonal_forecast(inflows, 0.5)),
    "skill 0.8": (tables[0.8], seasonal_forecast(inflows, 0.8)),
    "perfect": (with_perfect, perfect),
}
chosen_plans = {name: winter_plan_within(table, 2.0) for name, (table, _) in cases.items()}
rationed = {
    name: int(np.sum(simulate_winter(inflows, chosen_plans[name], forecast=forecast).winter_delivered_by_year < WINTER_FIELDS_TOTAL_MM3 - 0.01))
    for name, (_, forecast) in cases.items()
}
say(
    f"- **The curves nest.** Each forecast's curve lies between \"storage alone\" and the perfect forecast, and the "
    f"better the forecast, the closer to the perfect one. No curve crosses below the perfect one: a wrong forecast "
    f"can at best be ignored, and the optimizer makes sure it is never worse than that.\n"
    f"- **How a plan copes with a wrong forecast.** Take the plans that hold the farm to 2 Mm³ a year. To reach that "
    f"safety, the plan without a forecast rations {rationed['storage alone']} of the hundred winters; with a forecast "
    f"of skill 0.5 it rations {rationed['skill 0.5']}, with skill 0.8 {rationed['skill 0.8']}, with the perfect "
    f"forecast {rationed['perfect']}. The weaker the forecast, the more winters the plan must hold back to be safe "
    f"against the summers the forecast gets wrong. Those extra rationed winters are the price of the forecast's "
    f"errors, and the optimizer pays it by moving the trigger: {chosen_plans['skill 0.5'].trigger:.0f} Mm³ of water "
    f"in sight with skill 0.5 against {chosen_plans['perfect'].trigger:.0f} with the perfect forecast.\n"
    f"- **The operator does not need to know the formula.** It needs to know the skill, and to choose the plan for "
    f"it. A plan tuned for a perfect forecast and fed a weak one would ration too rarely; that is the trap the next "
    f"figure quantifies.",
    f"- **Egri chiziqlar ichma-ich joylashadi.** Har bir prognozning egri chizigʻi \"faqat ombordagi suv\" bilan "
    f"mukammal prognoz orasida yotadi, va prognoz qancha yaxshi boʻlsa, mukammalga shuncha yaqin. Hech bir egri "
    f"chiziq mukammalnikidan pastga kesib oʻtmaydi: notoʻgʻri prognozni eng yaxshi holda eʼtiborsiz qoldirish "
    f"mumkin, optimallashtirgich esa undan yomon boʻlmasligini taʼminlaydi.\n"
    f"- **Reja notoʻgʻri prognoz bilan qanday kurashadi.** Xoʻjalikni yiliga 2 mln m³ da ushlab turadigan rejalarni "
    f"oling. Shu xavfsizlikka erishish uchun prognozsiz reja yuz qishdan {rationed['storage alone']} tasini "
    f"cheklaydi; sifati 0,5 prognoz bilan {rationed['skill 0.5']} tasini, 0,8 bilan {rationed['skill 0.8']} tasini, "
    f"mukammal prognoz bilan {rationed['perfect']} tasini. Prognoz qancha kuchsiz boʻlsa, reja prognoz xato "
    f"qiladigan yozlardan xavfsiz boʻlish uchun shuncha koʻp qishda ushlab turishi kerak. Bu qoʻshimcha cheklangan "
    f"qishlar — prognoz xatolarining narxi, optimallashtirgich uni chegarani surish bilan toʻlaydi: sifat 0,5 da "
    f"koʻrinib turgan suv {chosen_plans['skill 0.5'].trigger:.0f} mln m³, mukammal prognozda "
    f"{chosen_plans['perfect'].trigger:.0f}.\n"
    f"- **Operator formulani bilishi shart emas.** U sifatni bilishi va unga mos rejani tanlashi kerak. Mukammal "
    f"prognozga moslangan, lekin kuchsiz prognoz berilgan reja juda kam cheklagan boʻlardi; keyingi rasm aynan shu "
    f"tuzoqni oʻlchaydi.",
    f"- **Кривые вложены друг в друга.** Кривая каждого прогноза лежит между «только запасом» и идеальным "
    f"прогнозом, и чем лучше прогноз, тем ближе к идеальному. Ни одна кривая не опускается ниже идеальной: ошибочный "
    f"прогноз в лучшем случае можно проигнорировать, и оптимизатор следит, чтобы хуже этого не стало.\n"
    f"- **Как план справляется с ошибочным прогнозом.** Возьмём планы, удерживающие хозяйство на 2 млн м³ в год. "
    f"Чтобы достичь этой безопасности, план без прогноза лимитирует {rationed['storage alone']} из ста зим; с "
    f"прогнозом качества 0,5 — {rationed['skill 0.5']}, с качеством 0,8 — {rationed['skill 0.8']}, с идеальным "
    f"прогнозом — {rationed['perfect']}. Чем слабее прогноз, тем больше зим план должен придерживать воду, чтобы "
    f"застраховаться от тех лет, где прогноз ошибается. Эти лишние лимитированные зимы — цена ошибок прогноза, и "
    f"оптимизатор платит её сдвигом порога: {chosen_plans['skill 0.5'].trigger:.0f} млн м³ воды на горизонте при "
    f"качестве 0,5 против {chosen_plans['perfect'].trigger:.0f} при идеальном прогнозе.\n"
    f"- **Оператору не нужно знать формулу.** Ему нужно знать качество и выбрать план под него. План, настроенный "
    f"на идеальный прогноз и получающий слабый, лимитировал бы слишком редко; именно эту ловушку измеряет следующий "
    f"рисунок.",
)

**What we find, on one chart.** Three curves are hard to compare by eye, so we reduce each to one number. We fix
the farm's safety at a summer shortage of 2 Mm³ a year, the point we have used throughout, and ask: with this
forecast, how much must the winter fields give up to keep the farm that safe? One point per skill, from "an
average summer every year" to the perfect forecast, with the operator without a forecast as the line to beat.

In [ ]:
# @title Code: fig-forecast-value
#| label: fig-forecast-value
#| fig-cap: "What the forecast is worth against its skill: the winter shortage left when the farm's shortage is held to 2 Mm³ a year. The orange line is the operator without a forecast."
curve = forecast_value(tables, summer_cap=2.0)
_ = plot_forecast_value(curve, no_forecast=at_2, summer_cap=2.0)

In [ ]:
# @title Code: forecast-value-text
#| label: forecast-value-text
#| echo: false
value_at = dict(zip(curve["skill"], curve["winter_shortage"]))
half_way = (at_2 - value_at[0.5]) / (at_2 - value_at[1.0]) * 100.0
say(
    f"- **A forecast with no skill is no forecast.** At skill 0 the forecast says 40 Mm³ every year, and the rule is "
    f"back to watching the storage: the curve starts on the orange line, within the grid's steps "
    f"({value_at[0.0]:.1f} against {at_2:.1f} Mm³ a year).\n"
    f"- **Value grows with skill, but not in proportion.** A forecast of skill 0.5, halfway to perfect, delivers "
    f"{half_way:.0f} % of what the perfect forecast delivers; one of skill 0.8 leaves the winter fields "
    f"{value_at[0.8]:.1f} Mm³ a year short, against {value_at[1.0]:.1f} for the perfect forecast and {at_2:.1f} "
    f"with none. Most of the value sits in the last stretch of skill. The curve wiggles because the forecast "
    f"errors are one random draw of a hundred years; the trend is what counts.\n"
    f"- **Where real forecasts sit.** The statistical forecasts that Apel et al. (2018) built for rivers of Central "
    f"Asia explain 60 to 80 % of the year-to-year variation of the April–September flow when issued in January, "
    f"and 68 to 97 % when issued on 1 April, the date that matters most for the region's water planning. In our "
    f"terms that is a skill of roughly 0.8 to 0.95: real forecasts sit on the right-hand part of the curve, where "
    f"the value is.\n"
    f"- **The forecast's worth depends on the decision, not only on the forecast.** The same forecast was worth "
    f"nothing for the summer decisions of this reservoir (we tried: the operator learns the summer by May, and "
    f"water saved in one month is still there the next), and worth {at_2 - value_at[1.0]:.1f} Mm³ a year for the "
    f"winter decision. A forecast is worth what the decision it informs can do with it, a point made for weather "
    f"forecasts by Murphy (1993) and for reservoirs by Anghileri et al. (2016).\n"
    f"- **What the operator does with a wrong forecast matters.** Here the rule takes the forecast at its word, and "
    f"the optimizer compensates by choosing the trigger. A cautious operator could count only part of the "
    f"forecast; exercise 5 asks whether that helps.\n"
    f"- **What a real forecast service adds.** Not one number but a range: an ensemble of possible summers, or a "
    f"best guess with an error band, and updates every month as the snow accumulates. The natural next step, not "
    f"taken here, is to count on a cautious quantile of that range (\"the summer we are 80 % sure of\") and let the "
    f"optimizer choose how cautious; and to let the forecast improve through the winter, as exercise 6 begins to "
    f"do.",
    f"- **Sifati yoʻq prognoz — prognoz emas.** Sifat 0 da prognoz har yili 40 mln m³ deydi, va qoida yana "
    f"ombordagi suvga qarashga qaytadi: egri chiziq toʻr qadamlari doirasida toʻq sariq chiziqdan boshlanadi "
    f"(yiliga {value_at[0.0]:.1f} ga qarshi {at_2:.1f} mln m³).\n"
    f"- **Qadr sifat bilan oʻsadi, lekin mutanosib emas.** Sifati 0,5 boʻlgan, mukammalga yarim yoʻl bosgan "
    f"prognoz mukammal prognoz beradiganning {half_way:.0f} % ini beradi; sifati 0,8 boʻlgani qishki dalalarni "
    f"yiliga {value_at[0.8]:.1f} mln m³ taqchil qoldiradi, mukammal prognozda {value_at[1.0]:.1f}, prognozsiz "
    f"{at_2:.1f}. Qadrning koʻp qismi sifatning oxirgi boʻlagida yotadi. Egri chiziq tebranadi, chunki prognoz "
    f"xatolari — yuz yillik bitta tasodifiy tanlanma; muhimi — umumiy yoʻnalish.\n"
    f"- **Haqiqiy prognozlar qayerda turadi.** Apel va boshq. (2018) Markaziy Osiyo daryolari uchun qurgan statistik "
    f"prognozlar aprel–sentyabr oqimining yildan yilga oʻzgarishining 60–80 % ini yanvarda berilganda, 68–97 % ini "
    f"esa 1 aprelda — mintaqa suv rejalashtiruvi uchun eng muhim sanada — berilganda tushuntiradi. Bizning "
    f"atamalarda bu taxminan 0,8 dan 0,95 gacha sifat: haqiqiy prognozlar egri chiziqning oʻng qismida, qadr "
    f"yotgan joyda turadi.\n"
    f"- **Prognozning qadri faqat prognozga emas, qarorga bogʻliq.** Xuddi shu prognoz bu omborning yozgi "
    f"qarorlari uchun hech narsaga arzimadi (biz sinab koʻrdik: operator yozni may oyigacha bilib oladi, bir oyda "
    f"saqlangan suv esa keyingi oyda ham joyida turadi), qishki qaror uchun esa yiliga "
    f"{at_2 - value_at[1.0]:.1f} mln m³ ga arzidi. Prognoz u xabardor qilgan qaror u bilan nima qila olsa, "
    f"shunchaga arziydi — bu fikrni ob-havo prognozlari uchun Murphy (1993), suv omborlari uchun Anghileri va "
    f"boshq. (2016) aytgan.\n"
    f"- **Operator notoʻgʻri prognoz bilan nima qilishi muhim.** Bu yerda qoida prognozga soʻzsiz ishonadi, "
    f"optimallashtirgich esa chegarani tanlab buni qoplaydi. Ehtiyotkor operator prognozning faqat bir qismini "
    f"hisobga olishi mumkin edi; 5-mashq bu yordam beradimi, deb soʻraydi.\n"
    f"- **Haqiqiy prognoz xizmati nima qoʻshadi.** Bitta raqam emas, oraliq: mumkin boʻlgan yozlar ansambli yoki xato "
    f"oraligʻi bilan eng yaxshi taxmin, va qor toʻplangani sari har oy yangilanish. Bu yerda qilinmagan tabiiy "
    f"keyingi qadam — shu oraliqning ehtiyotkor kvantiliga (\"80 % ishonchimiz komil boʻlgan yoz\") tayanish va "
    f"qanchalik ehtiyotkor boʻlishni optimallashtirgichga tanlatish; hamda prognozni qish davomida yaxshilanib "
    f"borishiga yoʻl qoʻyish, 6-mashq buni boshlaydi.",
    f"- **Прогноз без качества — не прогноз.** При качестве 0 прогноз каждый год говорит 40 млн м³, и правило снова "
    f"смотрит только на запас: кривая начинается на оранжевой линии, в пределах шага сетки "
    f"({value_at[0.0]:.1f} против {at_2:.1f} млн м³ в год).\n"
    f"- **Ценность растёт с качеством, но не пропорционально.** Прогноз качества 0,5, на полпути к идеальному, даёт "
    f"{half_way:.0f} % того, что даёт идеальный; прогноз качества 0,8 оставляет зимним полям дефицит "
    f"{value_at[0.8]:.1f} млн м³ в год против {value_at[1.0]:.1f} у идеального и {at_2:.1f} без прогноза. "
    f"Большая часть ценности лежит на последнем отрезке качества. Кривая колеблется, потому что ошибки прогноза — "
    f"одна случайная выборка из ста лет; важна тенденция.\n"
    f"- **Где находятся настоящие прогнозы.** Статистические прогнозы, которые Apel et al. (2018) построили для рек "
    f"Центральной Азии, объясняют 60–80 % межгодовой изменчивости стока за апрель–сентябрь при выпуске в январе и "
    f"68–97 % при выпуске 1 апреля — в дату, которая важнее всего для водного планирования региона. В наших "
    f"терминах это качество примерно от 0,8 до 0,95: настоящие прогнозы находятся на правой части кривой, там, "
    f"где и лежит ценность.\n"
    f"- **Ценность прогноза зависит от решения, а не только от прогноза.** Тот же прогноз ничего не стоил для "
    f"летних решений этого водохранилища (мы проверяли: оператор узнаёт лето к маю, а вода, сбережённая в одном "
    f"месяце, никуда не девается к следующему) и стоил {at_2 - value_at[1.0]:.1f} млн м³ в год для зимнего "
    f"решения. Прогноз стоит столько, сколько может сделать с ним решение, которое он обслуживает, — мысль, "
    f"высказанная для прогнозов погоды Murphy (1993), а для водохранилищ — Anghileri et al. (2016).\n"
    f"- **Важно, что оператор делает с ошибочным прогнозом.** Здесь правило верит прогнозу на слово, а "
    f"оптимизатор компенсирует это выбором порога. Осторожный оператор мог бы учитывать лишь часть прогноза; "
    f"упражнение 5 спрашивает, помогает ли это.\n"
    f"- **Что добавляет настоящая прогностическая служба.** Не одно число, а диапазон: ансамбль возможных лет или "
    f"лучшую оценку с полосой ошибки, и обновления каждый месяц по мере накопления снега. Естественный следующий "
    f"шаг, здесь не сделанный, — опираться на осторожный квантиль этого диапазона («лето, в котором мы уверены на "
    f"80 %») и дать оптимизатору выбрать меру осторожности; и позволить прогнозу уточняться по ходу зимы, с чего "
    f"начинает упражнение 6.",
)

**In one sentence.** A forecast that is sometimes wrong is still worth acting on, but the operator must act on it
with a plan chosen for its quality, and the better the forecast, the more it is worth, with most of the value in
the last stretch between a good forecast and a perfect one: which is exactly where the real forecasts of the
region's hydrometeorological services sit, and why their quality matters for a reservoir like this one.

# Another river: the paper's regime

The paper's river peaks in winter and is lowest in summer, the opposite of ours. Same reservoir, same farm, same
hundred random draws; only the seasons are swapped. Does the trade-off look the same?

In [ ]:
# @title Code: fig-regimes
#| label: fig-regimes
#| fig-cap: "Left: the average month of both rivers against the farm's need. Right: the unbeaten plans of both. Same reservoir, same need, same random seed."
paper_inflows = synthetic_inflows(regime=PAPER_REGIME)  # the paper's seasons: winter 40 Mm³, summer 25 Mm³
paper_table = score_plans(paper_inflows, plan_grid())  # the same 441 plans on that river

_ = plot_regime_comparison(
    {"Central Asian regime (this notebook)": (inflows, table), "Paper regime (winter peak)": (paper_inflows, paper_table)}
)

In [ ]:
# @title Code: regimes-text
#| label: regimes-text
#| echo: false
paper_front = front_of(paper_table)
paper_base = simulate_plan(paper_inflows, NO_RATIONING).scores
ours_50 = front[front["worst_month"] <= 50.0].iloc[0]
paper_50 = paper_front[paper_front["worst_month"] <= 50.0].iloc[0]
say(
    f"- **How to read the figure.** Left: the average inflow of each month, blue for our river, orange for the "
    f"paper's; the black step is the farm's need, 0.83 Mm³ a month in winter and 7.5 in summer. Our summer bars "
    f"reach almost up to the need; the paper's summer bars stay far below it, so its reservoir has to carry most "
    f"of the summer water over from winter. Right: the unbeaten plans of both rivers, on the chart you know.\n"
    f"- **Same shape.** Both curves bend the same way: cheap at first, expensive at the end. The example works for "
    f"either river.\n"
    f"- **The paper's river is the harder case.** Without rationing it is {paper_base.worst_month:.0f} % short in "
    f"its worst month (ours: {s.worst_month:.0f} %), and holding the worst month to 50 % costs "
    f"{paper_50['mean_shortage'] - paper_base.mean_shortage:.1f} Mm³ per year there against "
    f"{ours_50['mean_shortage'] - s.mean_shortage:.1f} here. With a summer peak the water arrives when the farm "
    f"needs it, and the reservoir only has to cover the gap.\n"
    f"- **What runs our reservoir dry** is a poor winter (little to store) followed by a weak melt. That is the "
    f"Central Asian pattern: the reservoir is filled from the low winter flow, as Kattakurgan is.",
    f"- **Rasmni qanday oʻqish.** Chapda: har bir oyning oʻrtacha oqimi, koʻk — bizning daryo, toʻq sariq — "
    f"maqoladagi daryo; qora pogʻona — xoʻjalik talabi, qishda oyiga 0,83 mln m³, yozda 7,5. Bizning yoz ustunlari "
    f"deyarli talabgacha yetadi; maqoladagi daryoning yoz ustunlari undan ancha pastda, shuning uchun uning ombori "
    f"yoz suvining koʻp qismini qishdan olib oʻtishi kerak. Oʻngda: ikkala daryoning yengilmagan rejalari, sizga "
    f"tanish diagrammada.\n"
    f"- **Shakli bir xil.** Ikkala egri chiziq ham bir xil bukiladi: avval arzon, oxirida qimmat. Misol ikkala daryo "
    f"uchun ham ishlaydi.\n"
    f"- **Maqoladagi daryo — qiyinroq holat.** Cheklashsiz uning eng yomon oyi {paper_base.worst_month:.0f} % "
    f"taqchil (bizniki: {s.worst_month:.0f} %), va eng yomon oyni 50 % da ushlab turish u yerda yiliga "
    f"{paper_50['mean_shortage'] - paper_base.mean_shortage:.1f} mln m³, bizda esa "
    f"{ours_50['mean_shortage'] - s.mean_shortage:.1f} mln m³ ga tushadi. Yozgi toshqin bilan suv xoʻjalikka kerak "
    f"paytda keladi, va ombor faqat farqni qoplashi kerak.\n"
    f"- **Bizning omborni quritadigan narsa** — kam suvli qish (saqlashga kam suv) va undan keyingi kuchsiz erish. "
    f"Bu Markaziy Osiyo manzarasi: ombor, Kattaqoʻrgʻon suv ombori kabi, kam suvli qishki oqimdan toʻldiriladi.",
    f"- **Как читать рисунок.** Слева: средний приток каждого месяца, синий — наша река, оранжевый — река из "
    f"статьи; чёрная ступенька — потребность хозяйства, 0,83 млн м³ в месяц зимой и 7,5 летом. Наши летние столбики "
    f"почти дотягивают до потребности; летние столбики реки из статьи остаются далеко ниже, так что её "
    f"водохранилищу приходится переносить большую часть летней воды с зимы. Справа: непревзойдённые планы обеих "
    f"рек на уже знакомом графике.\n"
    f"- **Та же форма.** Обе кривые изгибаются одинаково: сначала дёшево, в конце дорого. Пример работает для обеих "
    f"рек.\n"
    f"- **Река из статьи — более трудный случай.** Без лимитирования её худший месяц имеет дефицит "
    f"{paper_base.worst_month:.0f} % (у нашей: {s.worst_month:.0f} %), а удержание худшего месяца на уровне 50 % "
    f"стоит там {paper_50['mean_shortage'] - paper_base.mean_shortage:.1f} млн м³ в год против "
    f"{ours_50['mean_shortage'] - s.mean_shortage:.1f} у нас. При летнем пике вода приходит тогда, когда она нужна "
    f"хозяйству, и водохранилищу остаётся лишь покрыть разницу.\n"
    f"- **Наше водохранилище опустошает** скудная зима (мало что накопить), за которой следует слабое таяние. Это "
    f"центральноазиатская картина: водохранилище наполняется из скудного зимнего стока, как Каттакурганское.",
)

# From this toy to a real reservoir

- **Real rules have the same knobs, many times over.** The release rule of the Kattakurgan reservoir in the
  Zarafshan model of this repository (`ZRBReleaseRule`) has a storage level below which releases are cut (`v1`) and
  a cut factor (`buffer_coef`): our trigger and ration, with a separate value for each month. That is why the
  full model needs the optimizer.
- **What this toy leaves out.** A forecast that is updated as the winter goes on, and the uncertainty band that
  comes with it; more users with different priorities; how crops actually respond to a shortage; evaporation that
  depends on the lake's surface area; return flows.
- **One river is one river.** The numbers here belong to this one synthetic series of a hundred years. Other
  series give other numbers. The shape (a bent curve from "no rationing" to "cut deep") was the same for the three
  other series we tested.

# Exercises

1. **A bigger reservoir.** In `rationing_coursebook.py`, set `CAPACITY_MM3 = 60`. Does rationing still pay? Where
   is the bend now?
2. **A different second goal.** Replace "worst month" with "years short" in `objectives()` and run the optimizer
   again. Which plans are unbeaten now? Is "no rationing" still one of them?
3. **A different river.** Use `synthetic_inflows(seed=1)`, then `synthetic_inflows(regime=PAPER_REGIME, seed=1)`.
   Which numbers change, and what stays the same?
4. **A seasonal rule.** Give the rule two rations, one for early and one for late summer. Does the optimizer find
   plans that beat the two-knob curve?
5. **A cautious operator.** In `WinterRelease`, count only part of the forecast: `node.storage + 0.7 * forecast`.
   Re-run the skill curve. Does caution help with a weak forecast (skill 0.5)? Does it hurt with a strong one?
6. **A late forecast.** Real forecasts arrive in late winter, when the snow has fallen. Let the rule use the
   forecast only in the last two winter months (`month >= 4`) and the storage alone before. How much of the
   forecast's value survives?

How to do an exercise in Colab: the code of every cell is folded; click "Show code" to see and change it, then run
the cells below it again. Exercise 1 asks you to change a file, which Colab does not let you do; instead, run this
in a new cell before the grid section, then run that section again:

```python
import zarafshan_taqsim.rationing_coursebook as m
m.CAPACITY_MM3 = 60.0
```

# Sources

- Hashimoto, T., Stedinger, J. R., Loucks, D. P. (1982). Reliability, resiliency, and vulnerability criteria for
  water resource system performance evaluation. *Water Resources Research* 18(1), 14–20. The reservoir, its
  numbers (p. 17, Table 1; seasons swapped here) and the trade-off between failing often and failing badly
  (Fig. 9).
- Draper, A. J., Lund, J. R. (2004). Optimal hedging and carryover storage value. *Journal of Water Resources
  Planning and Management* 130(1), 83–87. When rationing pays (p. 83) and why a rule with two parameters is often
  enough (p. 87).
- Apel, H., Abdykerimova, Z., Agalhanova, M., Baimaganbetov, A., Gavrilenko, N., Gerlitz, L., Kalashnikova, O.,
  Unger-Shayesteh, K., Vorogushyn, S., Gafurov, A. (2018). Statistical forecast of seasonal discharge in Central
  Asia using observational records: development of a generic linear modelling tool for operational water resource
  management. *Hydrology and Earth System Sciences* 22, 2225–2254. How the region's hydrometeorological services
  forecast the April–September flow, and from what.
- Anghileri, D., Voisin, N., Castelletti, A., Pianosi, F., Nijssen, B., Lettenmaier, D. P. (2016). Value of
  long-term streamflow forecasts to reservoir operations for water supply in snow-dominated river catchments.
  *Water Resources Research* 52, 4209–4225. Forecast value measured against the perfect forecast, for a reservoir.
- Murphy, A. H. (1993). What is a good forecast? An essay on the nature of goodness in weather forecasting.
  *Weather and Forecasting* 8, 281–293. The distinction between a forecast's quality and its value to a decision.
- Deb, K., Pratap, A., Agarwal, S., Meyarivan, T. (2002). A fast and elitist multiobjective genetic algorithm:
  NSGA-II. *IEEE Transactions on Evolutionary Computation* 6(2), 182–197. The search method behind
  `taqsim.optimize`.